# MMDS Phase 3 Spark SQL analytics

Read the verified Phase 2 saved output and emit compact aggregate evidence.

In [ ]:
import importlib.util
from importlib.metadata import PackageNotFoundError, version
import os
import platform
import subprocess
import sys
from pathlib import Path

print(f"Python {platform.python_version()}", flush=True)
try:
    installed = version("pyspark")
except PackageNotFoundError:
    installed = None
if installed != "4.3.0":
    subprocess.run([sys.executable, "-m", "pip", "install", "pyspark==4.3.0"], check=True)
if importlib.util.find_spec("pyarrow") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "pyarrow"], check=True)
subprocess.run(["java", "-version"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable
if importlib.util.find_spec('matplotlib') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'matplotlib'], check=True)
os.environ['MPLCONFIGDIR'] = '/kaggle/temp/mmds_phase3/matplotlib'


In [ ]:
project = Path('/kaggle/temp/mmds_phase3/code')
sources = {'src/ingest/dataset_audit.py': '"""Measure the MMDS source without changing or exporting traffic records."""\n\nimport argparse\nfrom collections import Counter\nimport csv\nfrom datetime import datetime\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport time\nimport unicodedata\n\n\nLABEL_NAMES = {"label", "traffic_label", "class", "attack_type"}\nTIME_NAMES = {"timestamp", "time_stamp", "flow_timestamp"}\nIDENTIFIER_NAMES = {"flow_id", "src_ip", "dst_ip", "source_ip", "destination_ip"}\nNONNEGATIVE = re.compile(r"duration|length|(?:^|_)len(?:_|$)|pkts?|packets?|bytes?|byts?|count|(?:^|_)iat(?:_|$)|flags?|active|idle|(?:^|_)rate(?:_|$)|(?:^|_)ps$")\nSENTINELS = {"infinity", "+infinity", "-infinity", "inf", "+inf", "-inf"}\nBENIGN = "BENIGN"\n# The quantized comparison is limited to these non-identity flow measurements.\nNEAR_FIELDS = {\n    "flow_duration", "tot_fwd_pkts", "tot_bwd_pkts",\n    "totlen_fwd_pkts", "totlen_bwd_pkts", "flow_iat_mean",\n}\n\n\ndef normalize_column(value):\n    value = unicodedata.normalize("NFKC", value).lstrip("\\ufeff").strip().casefold()\n    return re.sub(r"[^\\w]+", "_", value, flags=re.UNICODE).strip("_")\n\n\ndef normalize_label(value):\n    return " ".join(unicodedata.normalize("NFKC", value).split()).casefold() if value is not None else None\n\n\ndef proposed_binary(value):\n    """Only BENIGN has a default mapping; observed attack families need review."""\n    return "benign" if normalize_label(value) == BENIGN.casefold() else "unmapped"\n\n\ndef capture_day(name):\n    match = re.search(r"-(\\d{2}-\\d{2}-\\d{4})_", name)\n    return datetime.strptime(match.group(1), "%d-%m-%Y").date().isoformat() if match else None\n\n\ndef inspect_schema(header):\n    names = [normalize_column(col) for col in header]\n    original_counts = Counter(header)\n    normalized_counts = Counter(names)\n    return {\n        "original": header,\n        "normalized": names,\n        "exact_duplicates": sorted(k for k, n in original_counts.items() if n > 1),\n        "normalized_duplicates": sorted(k for k, n in normalized_counts.items() if n > 1),\n        "whitespace_columns": [i for i, raw in enumerate(header) if raw != raw.strip()],\n        "encoding_columns": [i for i, raw in enumerate(header) if any(ord(c) > 127 for c in raw)],\n        "unsupported_indices": [i for i, name in enumerate(names) if not name or normalized_counts[name] > 1],\n        "label_indices": [i for i, name in enumerate(names) if name in LABEL_NAMES],\n        "timestamp_indices": [i for i, name in enumerate(names) if name in TIME_NAMES],\n    }\n\n\ndef compare_schema(reference, candidate):\n    left, right = reference["normalized"], candidate["normalized"]\n    return {\n        "same_order": left == right,\n        "same_original_header": reference["original"] == candidate["original"],\n        "missing": list((Counter(left) - Counter(right)).elements()),\n        "extra": list((Counter(right) - Counter(left)).elements()),\n        "compatible": Counter(left) == Counter(right)\n        and not reference["unsupported_indices"] and not candidate["unsupported_indices"],\n    }\n\n\ndef read_inventory(root, config):\n    root = Path(root)\n    expected = {f\'{config["directory"]}/{name}\': size for name, size in config["files"].items()}\n    available = {p.relative_to(root).as_posix(): p for p in root.rglob("*") if p.is_file()}\n    names = Counter(path.rsplit("/", 1)[-1] for path in available)\n    items = []\n    for rel in sorted(expected.keys() | available.keys()):\n        path = available.get(rel)\n        name = rel.rsplit("/", 1)[-1]\n        size = None\n        access = "missing"\n        if path is not None:\n            try:\n                size = path.stat().st_size\n                with path.open("rb") as source:\n                    source.read(1)\n                access = "readable"\n            except OSError:\n                access = "unreadable"\n        items.append({\n            "path": rel, "name": name, "capture_day": capture_day(name) if rel in expected else None,\n            "bytes": size, "expected_bytes": expected.get(rel), "access": access,\n            "unexpected": rel not in expected, "duplicate_basename": names[name] > 1,\n            "size_matches": size == expected[rel] if rel in expected else None,\n        })\n    return items\n\n\ndef counts_reconcile(counts, labels=None):\n    return (\n        counts["physical_lines"] == sum(counts[key] for key in ("blank", "headers", "malformed", "parsed"))\n        and counts["malformed"] == sum(counts[key] for key in ("bad_width_or_quotes", "spark_corrupt"))\n        and counts["data_records"] == counts["parsed"] + counts["malformed"]\n        and (labels is None or sum(row["count"] for row in labels) == counts["parsed"])\n    )\n\n\ndef stable_sample(raw, seed, modulus):\n    from pyspark.sql import functions as F\n\n    return F.pmod(F.xxhash64(F.lit(seed), raw), F.lit(modulus)) == 0\n\n\ndef timestamp_columns(raw, formats):\n    from pyspark.sql import functions as F\n\n    value = F.trim(raw)\n    primary_options = [F.try_to_timestamp(value, F.lit(fmt)) for fmt in formats]\n    shadow_options = [\n        F.try_to_timestamp(value, F.lit(fmt.replace("d/M/yyyy", "M/d/yyyy")))\n        for fmt in formats if fmt.startswith("d/M/yyyy")\n    ]\n    primary = F.coalesce(*primary_options)\n    shadow = F.coalesce(*shadow_options)\n    matched = F.coalesce(*[F.when(expr.isNotNull(), F.lit(fmt)) for fmt, expr in zip(formats, primary_options)])\n    return primary, shadow, matched\n\n\ndef numeric_fields(value):\n    from pyspark.sql import functions as F\n\n    stripped = F.trim(value)\n    present = value.isNotNull() & (stripped != "")\n    number = stripped.try_cast("double")\n    nan = present & F.isnan(number)\n    inf = present & (\n        F.lower(stripped).isin(*sorted(SENTINELS)) | F.coalesce(F.abs(number) == float("inf"), F.lit(False))\n    )\n    finite = present & number.isNotNull() & ~nan & ~inf\n    failed = present & number.isNull() & ~inf\n    return present, number, nan, inf, finite, failed\n\n\ndef atomic_json(path, value):\n    stage = path.with_suffix(".tmp")\n    stage.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str) + "\\n", encoding="utf-8")\n    stage.replace(path)\n\n\ndef spark_session(config, scratch):\n    from pyspark.sql import SparkSession\n\n    spark_config = config["spark"]\n    os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")\n    return (\n        SparkSession.builder.appName("mmds-phase-1-dataset-audit")\n        .master(spark_config["master"])\n        .config("spark.driver.memory", spark_config["driver_memory"])\n        .config("spark.local.dir", str(scratch))\n        .config("spark.sql.shuffle.partitions", spark_config["shuffle_partitions"])\n        .config("spark.sql.files.maxPartitionBytes", spark_config["input_partition_bytes"])\n        .config("spark.sql.files.openCostInBytes", 4194304)\n        .config("spark.sql.adaptive.enabled", "true")\n        .config("spark.sql.session.timeZone", "UTC")\n        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")\n        .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")\n        .config("spark.sql.csv.parser.columnPruning.enabled", "false")\n        .config("spark.ui.enabled", "false")\n        .getOrCreate()\n    )\n\n\ndef record_frame(spark, path, header_line, width):\n    from pyspark.sql import functions as F\n\n    raw = F.col("raw")\n    blank = raw.rlike(r"^\\s*$")\n    header = raw == header_line\n    comma_width = F.length(raw) - F.length(F.replace(raw, F.lit(","), F.lit(""))) + 1\n    quoted = F.instr(raw, \'"\') > 0\n    field = r\'(?:"(?:[^"]|"")*"|[^",]*)\'\n    quote_grammar = "^" + field + "(?:," + field + ")*$"\n    masked = F.regexp_replace(raw, r\'"(?:[^"]|"")*"\', \'""\')\n    quoted_width = F.length(masked) - F.length(F.replace(masked, F.lit(","), F.lit(""))) + 1\n    valid = F.when(quoted, raw.rlike(quote_grammar) & (quoted_width == width)).otherwise(comma_width == width)\n    status = F.when(blank, "blank").when(header, "headers").when(~valid, "bad_width_or_quotes")\n    ddl = ", ".join([f"c{i} STRING" for i in range(width)] + ["_corrupt STRING"])\n    options = {\n        "mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt",\n        "quote": \'"\', "escape": \'"\', "nullValue": "\\u0000", "ignoreLeadingWhiteSpace": "false",\n        "ignoreTrailingWhiteSpace": "false", "unescapedQuoteHandling": "RAISE_ERROR",\n    }\n    physical = spark.read.text(str(path)).toDF("raw")\n    parsed = physical.withColumn("csv", F.when(status.isNull(), F.from_csv(raw, ddl, options)))\n    kind = F.coalesce(status, F.when(F.col("csv._corrupt").isNotNull(), "spark_corrupt"), F.lit("parsed"))\n    return parsed.select("raw", kind.alias("kind"), *[F.col(f"csv.c{i}").alias(f"c{i}") for i in range(width)])\n\n\ndef count_records(frame):\n    counts = {row["kind"]: row["count"] for row in frame.groupBy("kind").count().collect()}\n    summary = {k: counts.get(k, 0) for k in ("blank", "headers", "bad_width_or_quotes", "spark_corrupt", "parsed")}\n    summary["malformed"] = summary["bad_width_or_quotes"] + summary["spark_corrupt"]\n    summary["data_records"] = summary["parsed"] + summary["malformed"]\n    summary["physical_lines"] = summary["data_records"] + summary["headers"] + summary["blank"]\n    summary["repeated_header_lines"] = max(0, summary["headers"] - 1)\n    return summary\n\n\ndef count_if(condition):\n    from pyspark.sql import functions as F\n\n    return F.count(F.when(condition, 1))\n\n\ndef profile_columns(frame, schema, config):\n    from pyspark.sql import functions as F\n\n    sampled = frame.where(stable_sample(F.col("raw"), config["sample_seed"], config["sample_modulus"]))\n    sample_count = sampled.count()\n    # Exact value counts within the small sample; collect only a row per column.\n    elements = [\n        F.struct(F.lit(i).alias("index"), F.col(f"c{i}").alias("value"))\n        for i in range(len(schema["original"]))\n    ]\n    sample_summary = (\n        sampled.select(F.explode(F.array(*elements)).alias("cell")).select("cell.*")\n        .where(F.col("value").isNotNull() & (F.trim("value") != ""))\n        .groupBy("index", "value").count()\n        .groupBy("index").agg(F.sum("count").alias("present"), F.max("count").alias("dominant"))\n    )\n    sample_counts = {row["index"]: (row["present"], row["dominant"]) for row in sample_summary.collect()}\n    profiles = []\n    # Limit the size of each aggregate projection and reuse the on-disk parse.\n    for start in range(0, len(schema["original"]), 24):\n        full = []\n        stop = min(start + 24, len(schema["original"]))\n        for i in range(start, stop):\n            col = F.col(f"c{i}")\n            present, number, nan, inf, finite, failed = numeric_fields(col)\n            measures = {\n                "null": count_if(col.isNull()), "empty_or_whitespace": count_if(col.isNotNull() & (F.trim(col) == "")),\n                "present": count_if(present), "finite_numeric": count_if(finite),\n                "nan": count_if(nan), "infinite": count_if(inf), "numeric_conversion_failures": count_if(failed),\n                "negative": count_if(finite & (number < 0)),\n                "lexical_min": F.min(F.when(present, col)), "lexical_max": F.max(F.when(present, col)),\n            }\n            full.extend(expr.alias(f"c{i}_{key}") for key, expr in measures.items())\n        full_result = frame.agg(*full).first().asDict()\n        for i in range(start, stop):\n            prefix = f"c{i}_"\n            row = {key[len(prefix):]: value for key, value in full_result.items() if key.startswith(prefix)}\n            low, high = row.pop("lexical_min"), row.pop("lexical_max")\n            row["constant_nonempty"] = low == high if row["present"] else None\n            row["sample_present"], top_count = sample_counts.get(i, (0, 0))\n            row["sample_dominant_count"] = top_count\n            row["sample_dominant_fraction"] = top_count / row["sample_present"] if row["sample_present"] else None\n            row["near_constant_sample"] = row["sample_dominant_fraction"] >= 0.99 if row["sample_present"] else None\n            name = schema["normalized"][i]\n            support = row["finite_numeric"] + row["nan"] + row["infinite"]\n            identifier = name in IDENTIFIER_NAMES\n            numeric = support > 0 and support / row["present"] >= 0.95\n            row["type_candidate"] = (\n                "label" if i in schema["label_indices"] else "timestamp" if i in schema["timestamp_indices"]\n                else "unsupported" if i in schema["unsupported_indices"]\n                else "identifier" if identifier else "categorical" if name in {"protocol", "protocol_number"}\n                else "numeric" if numeric else "categorical" if row["present"] else "unobserved"\n            )\n            row["invalid_negative"] = (\n                row["negative"] if numeric and NONNEGATIVE.search(name) and not name.startswith("init_") else None\n            )\n            if row["type_candidate"] != "numeric":\n                row["numeric_conversion_failures"] = None\n            profiles.append(dict(row, index=i, original=schema["original"][i], normalized=name, sample_rows=sample_count))\n    return profiles\n\n\ndef summarize_time(frame, schema, formats, day):\n    from pyspark.sql import functions as F\n\n    results = []\n    for i in schema["timestamp_indices"]:\n        raw = F.col(f"c{i}")\n        primary, shadow, matched = timestamp_columns(raw, formats)\n        stamped = frame.select(raw.alias("source"), primary.alias("parsed"), shadow.alias("shadow"), matched.alias("format"))\n        parsed, shadow, source = F.col("parsed"), F.col("shadow"), F.col("source")\n        present = source.isNotNull() & (F.trim(source) != "")\n        date = F.to_date(parsed)\n        aggregates = stamped.agg(\n            F.count("*").alias("rows"), count_if(~present).alias("missing"), count_if(present & parsed.isNull()).alias("invalid"),\n            count_if(parsed.isNotNull()).alias("parsed"), F.min(parsed).alias("earliest"), F.max(parsed).alias("latest"),\n            count_if(shadow.isNotNull() & parsed.isNull()).alias("mdy_only"),\n            count_if(shadow.isNotNull() & parsed.isNotNull() & (F.to_date(shadow) != date)).alias("ambiguous"),\n            count_if(date == F.lit(day)).alias("date_matches"),\n            count_if(date.isNotNull() & (date != F.lit(day))).alias("date_mismatches"),\n            count_if(F.to_date(shadow) == F.lit(day)).alias("mdy_date_matches"),\n        ).first().asDict()\n        dates = [row.asDict() for row in stamped.where(parsed.isNotNull()).groupBy(date.alias("date"), "format").count().limit(501).collect()]\n        if len(dates) <= 500:\n            aggregates["date_formats"] = sorted(dates, key=lambda row: (row["date"], row["format"]))\n        results.append(dict(aggregates, index=i, original=schema["original"][i]))\n    return results\n\n\ndef fingerprint_frame(frame, schema, path, config):\n    from pyspark.sql import functions as F\n\n    label = schema["label_indices"]\n    if len(label) != 1:\n        return None\n    columns = sorted(\n        (name, i) for i, name in enumerate(schema["normalized"])\n        if name not in LABEL_NAMES | TIME_NAMES | IDENTIFIER_NAMES and i not in schema["unsupported_indices"]\n    )\n    schema_key = hashlib.sha256(json.dumps([name for name, _ in columns]).encode()).hexdigest()[:16]\n    selected = [(name, i) for name, i in columns if name in NEAR_FIELDS]\n    raw = F.col("raw")\n    near = F.lit(None).cast("string")\n    if len(selected) >= 3:\n        buckets = [F.floor(F.col(f"c{i}").try_cast("double") / (1000 if "duration" in name or "iat" in name else 10))\n                   for name, i in selected]\n        complete = F.lit(True)\n        for bucket in buckets:\n            complete = complete & bucket.isNotNull()\n        near = F.when(stable_sample(raw, config["sample_seed"], config["sample_modulus"]) & complete,\n                      F.sha2(F.to_json(F.array(*buckets)), 256))\n    return frame.select(\n        F.sha2(raw, 256).alias("row_hash"),\n        F.sha2(F.to_json(F.array(*[F.col(f"c{i}") for _, i in columns])), 256).alias("feature_hash"),\n        F.col(f"c{label[0]}").alias("label"), near.alias("near_hash"),\n        F.lit(schema_key).alias("schema_key"), F.lit(path).alias("source"),\n    )\n\n\ndef summarize_fingerprints(spark, paths):\n    from pyspark.sql import functions as F\n\n    rows = spark.read.parquet(*[str(path) for path in paths])\n    duplicate_groups = rows.groupBy("row_hash").count().where(F.col("count") > 1)\n    duplicates = duplicate_groups.agg(F.count("*").alias("groups"), F.sum(F.col("count") - 1).alias("excess")).first().asDict()\n    known = rows.where(F.col("label").isNotNull() & (F.trim(F.col("label")) != ""))\n    conflicts = (known.groupBy("schema_key", "feature_hash")\n                 .agg(F.min("label").alias("first"), F.max("label").alias("last"), F.count("*").alias("rows"))\n                 .where(F.col("first") != F.col("last"))\n                 .agg(F.count("*").alias("groups"), F.sum("rows").alias("rows")).first().asDict())\n    near_sample = rows.where(F.col("near_hash").isNotNull())\n    near = (near_sample.groupBy("schema_key", "near_hash")\n            .agg(F.min("feature_hash").alias("first"), F.max("feature_hash").alias("last"), F.count("*").alias("rows"))\n            .where(F.col("first") != F.col("last"))\n            .agg(F.count("*").alias("groups"), F.sum("rows").alias("rows")).first().asDict())\n    near["eligible_sample_rows"] = near_sample.count()\n    return {\n        "raw_line_duplicates_sha256": {key: value or 0 for key, value in duplicates.items()},\n        "label_conflicting_features_sha256": {key: value or 0 for key, value in conflicts.items()},\n        "near_sample_buckets": {key: value or 0 for key, value in near.items()},\n    }\n\n\ndef exact_duplicate_summary(spark, root, inventory):\n    """Group full parsed string arrays within identical original headers."""\n    from pyspark.sql import functions as F\n\n    schemas = {}\n    complete = []\n    for item in inventory:\n        if item["unexpected"] or item["access"] != "readable" or not item["size_matches"]:\n            raise ValueError("Exact duplicate measurement requires the verified inventory.")\n        path = root / item["path"]\n        with path.open("rb") as source:\n            header_line = source.readline(65536).decode("utf-8").rstrip("\\r\\n").lstrip("\\ufeff")\n        header = tuple(next(csv.reader([header_line], strict=True)))\n        schema_key = schemas.setdefault(header, len(schemas))\n        frame = record_frame(spark, path, header_line, len(header))\n        complete.append(frame.where(F.col("kind") == "parsed").select(\n            F.lit(schema_key).alias("schema"),\n            F.array(*[F.col(f"c{i}") for i in range(len(header))]).alias("values"),\n            F.lit(item["capture_day"]).alias("day"),\n        ))\n    combined = complete[0]\n    for frame in complete[1:]:\n        combined = combined.unionByName(frame)\n    groups = combined.groupBy("schema", "values").agg(\n        F.count("*").alias("count"), F.min("day").alias("first_day"), F.max("day").alias("last_day"),\n    )\n    result = groups.agg(\n        F.sum("count").alias("parsed_rows"),\n        F.count("*").alias("distinct_rows"),\n        count_if(F.col("count") > 1).alias("duplicate_groups"),\n        F.sum(F.col("count") - 1).alias("excess_copies"),\n        count_if(F.col("first_day") != F.col("last_day")).alias("cross_day_groups"),\n    ).first().asDict()\n    return result | {"files": len(complete), "original_schemas": len(schemas)}\n\n\ndef audit_exact_duplicates(config, root, output, scratch):\n    root, output, scratch = map(Path, (root, output, scratch))\n    if not root.is_dir() or any(path.resolve().is_relative_to(root.resolve()) for path in (output, scratch)):\n        raise ValueError("Use a mounted dataset root and separate writable output and scratch directories")\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    inventory = read_inventory(root, config)\n    atomic_json(output / "inventory.json", {\n        "dataset": {key: config[key] for key in ("owner", "slug", "version")},\n        "expected_bytes": sum(config["files"].values()),\n        "actual_total_bytes": sum(item["bytes"] or 0 for item in inventory),\n        "files": inventory,\n    })\n    spark = spark_session(config, scratch)\n    spark.sparkContext.setLogLevel("WARN")\n    try:\n        started = time.monotonic()\n        print("Measuring exact duplicate field arrays across the mounted captures.", flush=True)\n        result = exact_duplicate_summary(spark, root, inventory)\n        result["elapsed_seconds"] = round(time.monotonic() - started, 3)\n        result["spark_version"] = spark.version\n        result["spark"] = config["spark"]\n        atomic_json(output / "exact_duplicates.json", result)\n        print(f"Exact duplicate aggregation completed in {result[\'elapsed_seconds\']} s", flush=True)\n    finally:\n        spark.stop()\n\n\ndef measure_file(spark, path, item, output, cache_dir, config, result):\n    from pyspark import StorageLevel\n    from pyspark.sql import functions as F\n\n    with path.open("rb") as source:\n        header_bytes = source.readline(65536)\n    if not header_bytes.endswith(b"\\n"):\n        raise ValueError("Missing header newline or header exceeds 64 KiB")\n    header_line = header_bytes.decode("utf-8").rstrip("\\r\\n")\n    schema = inspect_schema(next(csv.reader([header_line.lstrip("\\ufeff")], strict=True)))\n    schema["utf8_bom"] = header_bytes.startswith(b"\\xef\\xbb\\xbf")\n    result["schema"] = schema\n    cache_path = cache_dir / (item["name"] + ".parquet")\n    stage = output / (item["name"] + ".json")\n    frame = record_frame(spark, path, header_line.lstrip("\\ufeff"), len(schema["original"])).persist(StorageLevel.DISK_ONLY)\n    try:\n        started = time.monotonic()\n        counts = count_records(frame)\n        if not counts_reconcile(counts) or not counts["headers"]:\n            raise ValueError("Header or row counts failed reconciliation")\n        result["counts"] = counts\n        result["stage_seconds"]["parse_and_count"] = round(time.monotonic() - started, 3)\n        atomic_json(stage, result)\n        valid = frame.where(F.col("kind") == "parsed")\n        started = time.monotonic()\n        if len(schema["label_indices"]) == 1:\n            label_col = F.col(f\'c{schema["label_indices"][0]}\')\n            labels = valid.groupBy(label_col.alias("label")).count().orderBy("label").limit(501).collect()\n            if len(labels) > 500:\n                raise ValueError("Label cardinality exceeds compact audit limit")\n            result["labels"] = [row.asDict() for row in labels]\n            if not counts_reconcile(counts, result["labels"]):\n                raise ValueError("Label counts do not reconcile")\n        result["timestamps"] = summarize_time(valid, schema, config["timestamp_formats"], item["capture_day"])\n        result["stage_seconds"]["labels_and_time"] = round(time.monotonic() - started, 3)\n        atomic_json(stage, result)\n        started = time.monotonic()\n        result["quality"] = profile_columns(valid, schema, config)\n        result["stage_seconds"]["quality"] = round(time.monotonic() - started, 3)\n        atomic_json(stage, result)\n        started = time.monotonic()\n        fingerprints = fingerprint_frame(valid, schema, item["path"], config)\n        if fingerprints is not None:\n            fingerprints.write.mode("overwrite").parquet(str(cache_path))\n            result["fingerprints"] = True\n        result["stage_seconds"]["fingerprints"] = round(time.monotonic() - started, 3)\n        return cache_path if fingerprints is not None else None\n    finally:\n        frame.unpersist(blocking=True)\n\n\ndef audit_dataset(config, root, output, scratch, only_days=None):\n    root, output, scratch = map(Path, (root, output, scratch))\n    if not root.is_dir() or any(path.resolve().is_relative_to(root.resolve()) for path in (output, scratch)):\n        raise ValueError("Use a mounted dataset root and separate writable output and scratch directories")\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    fingerprints_dir = scratch / "fingerprints"\n    fingerprints_dir.mkdir(exist_ok=True)\n    (scratch / "spark").mkdir(exist_ok=True)\n    inventory = read_inventory(root, config)\n    atomic_json(output / "inventory.json", {\n        "dataset": {key: config[key] for key in ("owner", "slug", "version")},\n        "expected_bytes": sum(config["files"].values()),\n        "actual_total_bytes": sum(item["bytes"] or 0 for item in inventory),\n        "files": inventory,\n    })\n    spark = spark_session(config, scratch / "spark")\n    spark.sparkContext.setLogLevel("WARN")\n    run = {\n        "spark": config["spark"] | {\n            "version": spark.version, "timezone": "UTC", "timestamp_type": "TIMESTAMP_NTZ",\n            "executor_cores": spark.sparkContext.defaultParallelism, "spark_application": spark.sparkContext.appName,\n        },\n        "timestamp_formats": config["timestamp_formats"], "files": {},\n    }\n    started = time.monotonic()\n    atomic_json(output / "run_summary.json", run)\n    fingerprints = []\n    try:\n        # Complete the smaller files first so expensive captures do not prevent partial findings.\n        for item in sorted(inventory, key=lambda entry: entry["bytes"] or 0):\n            if item["unexpected"] or (only_days is not None and item["capture_day"] not in only_days):\n                continue\n            name = item["name"]\n            result = {"name": name, "stage_seconds": {}, "status": "partial"}\n            stage = output / (name + ".json")\n            print(f"Audit start {name}", flush=True)\n            try:\n                if item["access"] != "readable" or not item["size_matches"]:\n                    raise ValueError("Missing, inaccessible or changed source file")\n                path = measure_file(spark, root / item["path"], item, output, fingerprints_dir, config, result)\n                if path is not None:\n                    fingerprints.append(path)\n                result["status"] = "complete"\n            except Exception as error:\n                result["error"] = f"{type(error).__name__}: {str(error)[:900]}"\n                print(result["error"], flush=True)\n            atomic_json(stage, result)\n            run["files"][name] = {"status": result["status"], "stage_seconds": result["stage_seconds"]}\n            atomic_json(output / "run_summary.json", run)\n            print(f"Audit {result[\'status\']} {name}: {result.get(\'counts\', {}).get(\'parsed\')} parsed", flush=True)\n\n        if fingerprints:\n            fingerprint_started = time.monotonic()\n            try:\n                run["fingerprint_summary"] = summarize_fingerprints(spark, fingerprints)\n                run["fingerprint_files"] = len(fingerprints)\n            except Exception as error:\n                run["fingerprint_error"] = f"{type(error).__name__}: {str(error)[:900]}"\n                print(run["fingerprint_error"], flush=True)\n            run["fingerprint_seconds"] = round(time.monotonic() - fingerprint_started, 3)\n    finally:\n        run["elapsed_seconds"] = round(time.monotonic() - started, 3)\n        atomic_json(output / "run_summary.json", run)\n        spark.stop()\n    return run\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--config", type=Path, default=Path("configs/dataset.yaml"))\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--scratch", type=Path, required=True)\n    parser.add_argument("--day", action="append", help="Audit only this ISO capture day; repeat for more days.")\n    parser.add_argument("--exact-duplicates", action="store_true", help="Run only the exact duplicate aggregation.")\n    args = parser.parse_args()\n    config = json.loads(args.config.read_text())\n    if args.exact_duplicates:\n        audit_exact_duplicates(config, args.root, args.output, args.scratch)\n    else:\n        audit_dataset(config, args.root, args.output, args.scratch, args.day)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/ingest/schema_contract.py': '"""Versioned schema, record identity and strict Phase 1 source reconciliation."""\n\nimport csv\nfrom dataclasses import dataclass\nimport json\nfrom pathlib import Path\n\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.dataset_audit import normalize_column, record_frame\n\n\nEXTRA_FIELDS = ("flow_id", "src_ip", "src_port", "dst_ip")\nTEXT_METADATA = ("flow_id", "src_ip", "dst_ip")\nSOURCE_COUNTS = ("parsed", "headers", "blank", "bad_width_or_quotes", "spark_corrupt")\nFEATURE_CONTEXT = ("record_fingerprint", "split_feature_key", "source_day", "temporal_eligibility")\nTARGETS = ("original_label", "normalized_label", "binary_label", "attack_family")\n\n\n@dataclass\nclass DataContract:\n    policy: dict\n\n    @classmethod\n    def load(cls, path):\n        return cls(json.loads(Path(path).read_text(encoding="utf-8")))\n\n    @property\n    def common_values(self):\n        return tuple(name for name in map(normalize_column, self.policy["headers"]["common_80"])\n                     if name not in {"label", "timestamp"})\n\n    @property\n    def feature_names(self):\n        return tuple(name for name in self.common_values if name not in self.policy["excluded_shared_features"])\n\n    def source_variant(self, source_day):\n        return "extended_84" if source_day == self.policy["extended_day"] else "common_80"\n\n    def verified_header(self, path, source_day):\n        variant = self.source_variant(source_day)\n        with path.open("rb") as source:\n            line = source.readline(65536)\n        if not line.endswith(b"\\n"):\n            raise ValueError(f"Missing or oversized header in {path.name}")\n        decoded = line.decode("utf-8-sig").rstrip("\\r\\n")\n        header = next(csv.reader([decoded], strict=True))\n        if header != self.policy["headers"][variant]:\n            raise ValueError(f"Unexpected original header/order in {path.name} (expected {variant})")\n        return variant, decoded, tuple(map(normalize_column, header))\n\n    def label_lookup(self):\n        # Match the exact audited originals. Normalization is for display only.\n        pairs = []\n        for original, values in self.policy["label_mapping"].items():\n            pairs.extend((F.lit(original), F.struct(\n                F.lit(values["normalized"]).alias("normalized"),\n                F.lit(values["binary"]).cast("byte").alias("binary"),\n                F.lit(values["family"]).alias("family"),\n            )))\n        return F.create_map(*pairs)\n\n\ndef source_records(spark, contract, path, source_day):\n    variant, line, names = contract.verified_header(path, source_day)\n    frame = record_frame(spark, path, line, len(names))\n    return variant, frame.select("raw", "kind", *[F.col(f"c{i}").alias(name) for i, name in enumerate(names)])\n\n\ndef record_key(raw, variant):\n    # Use exact UTF-8 source-line text (excluding line break), scoped by header variant.\n    return F.unhex(F.sha2(F.concat(F.lit(variant + "\\u0000"), raw), 256))\n\n\ndef reconcile_source(observed, expected):\n    for name in SOURCE_COUNTS:\n        if observed[name] != expected[name]:\n            raise ValueError(f"Source count mismatch for {name}: {observed[name]} vs {expected[name]}")\n    if observed["physical_lines"] != expected["physical_lines"]:\n        raise ValueError("Source physical lines do not reconcile")\n    return observed | {"repeated_headers": max(0, observed["headers"] - 1)}\n\n\ndef safe_union(frames):\n    """Align only frames that already have matching canonical names and column types."""\n    result = frames[0]\n    expected = {field.name: field.dataType for field in result.schema}\n    for frame in frames[1:]:\n        actual = {field.name: field.dataType for field in frame.schema}\n        if actual != expected:\n            raise ValueError("Canonical frames have different columns or types")\n        result = result.unionByName(frame)\n    return result\n', 'src/ingest/cleaning.py': '"""Phase 2 value-level diagnostics and policy transformations."""\n\nfrom pyspark.sql import Window, functions as F\n\nfrom src.ingest.dataset_audit import NONNEGATIVE, SENTINELS, timestamp_columns\nfrom src.ingest.schema_contract import EXTRA_FIELDS, FEATURE_CONTEXT, TARGETS, TEXT_METADATA, record_key\n\n\ndef issue_rule(item):\n    text, value = item["text"], item["value"]\n    missing = text.isNull() | (F.trim(text) == "")\n    inf = F.lower(F.trim(text)).isin(*sorted(SENTINELS)) | F.coalesce(F.abs(value) == float("inf"), F.lit(False))\n    return (\n        F.when(missing, "source_null")\n        .when(F.isnan(value), "nan")\n        .when(inf & (F.trim(text).startswith("-") | (value < 0)), "negative_infinity")\n        .when(inf, "positive_infinity")\n        .when(value.isNull(), "cast_failure")\n        .when(F.array_contains(item["sentinels"], value), "sentinel")\n        .when(item["nonnegative"] & (value < 0), "invalid_negative")\n    )\n\n\ndef canonical_frame(frame, contract, variant, source_name, source_day, timestamp_formats):\n    numeric_names = tuple(name for name in (*contract.common_values, "src_port") if name in frame.columns)\n    raw_values = F.array(*[F.col(name) for name in numeric_names])\n    config = contract.policy\n    policies = config["numeric_actions"]\n    policy_pairs = [F.lit(v) for pair in policies.items() for v in pair]\n    actions = F.create_map(*policy_pairs)\n    descriptors = F.array(*[\n        F.struct(F.lit(name).alias("field"), F.lit(bool(NONNEGATIVE.search(name) and not name.startswith("init_"))).alias("nonnegative"),\n                 F.array(*[F.lit(value).cast("double") for value in config["sentinel_values"].get(name, [])])\n                 .cast("array<double>").alias("sentinels"))\n        for name in numeric_names\n    ])\n    parsed, _, _ = timestamp_columns(F.col("timestamp"), timestamp_formats)\n    base = (\n        frame.where(F.col("kind") == "parsed")\n        .withColumn("_text", raw_values)\n        .withColumn("_numbers", F.transform("_text", lambda value: F.trim(value).try_cast("double")))\n        .withColumn("_rules", descriptors)\n        .withColumn("_tokens", F.zip_with(F.arrays_zip("_text", "_numbers"), "_rules",\n                                         lambda x, r: F.struct(r["field"].alias("field"), x["_text"].alias("text"),\n                                                              x["_numbers"].alias("value"), r["nonnegative"],\n                                                              r["sentinels"])))\n        .withColumn("_issues", F.transform("_tokens", lambda token: F.struct(\n            token["field"], token["text"].alias("source_text"), issue_rule(token).alias("reason"),\n        )))\n        .withColumn("quality_issues", F.transform(F.filter("_issues", lambda item: item["reason"].isNotNull()),\n                                                 lambda item: F.struct(item["field"], item["source_text"], item["reason"],\n                                                                       actions[item["reason"]].alias("action"))))\n        .withColumn("record_fingerprint", record_key(F.col("raw"), variant))\n        .withColumn("_label", contract.label_lookup()[F.col("label")])\n        .withColumn("parsed_event_time", parsed)\n    )\n    match = F.to_date("parsed_event_time") == F.lit(source_day).cast("date")\n    status = (F.when(F.col("timestamp").isNull() | (F.trim("timestamp") == ""), "missing")\n              .when(F.col("parsed_event_time").isNull(), "unparseable")\n              .when(match, "matched")\n              .otherwise("outside_capture_day"))\n    fields = {name: F.col("_numbers")[i].alias(name) for i, name in enumerate(numeric_names)}\n    for name in EXTRA_FIELDS:\n        if name in TEXT_METADATA:\n            fields[name] = (F.col(name) if name in frame.columns else F.lit(None).cast("string")).alias(name)\n        elif name not in fields:\n            fields[name] = F.lit(None).cast("double").alias(name)\n    missing_fields = [name for name in EXTRA_FIELDS if name not in frame.columns]\n    return base.select(\n        *fields.values(),\n        F.col("label").alias("original_label"), F.col("_label.normalized").alias("normalized_label"),\n        F.col("_label.binary").alias("binary_label"), F.col("_label.family").alias("attack_family"),\n        F.col("timestamp").alias("original_timestamp"), "parsed_event_time",\n        F.lit(source_name).alias("source_file"), F.lit(source_day).alias("source_day"),\n        "record_fingerprint", F.lit(variant).alias("schema_variant"),\n        F.lit(config["version"]).alias("cleaning_policy_version"),\n        F.lit(config["dataset_version"]).cast("int").alias("dataset_version"),\n        status.alias("timestamp_parse_status"), match.alias("timestamp_capture_day_match"),\n        F.coalesce(match, F.lit(False)).alias("temporal_eligibility"),\n        F.array(*[F.lit(name) for name in missing_fields]).cast("array<string>").alias("structural_missing_fields"),\n        "quality_issues",\n    )\n\n\ndef clean_frame(canonical, contract):\n    null_fields = F.transform(F.filter("quality_issues", lambda issue: issue["action"] == "nullify"),\n                              lambda issue: issue["field"])\n    base = canonical.withColumn("_null_fields", null_fields)\n    numeric = {*contract.common_values, "src_port"}\n    return base.select(*[\n        F.when(F.array_contains("_null_fields", name), F.lit(None).cast("double"))\n        .otherwise(F.col(name)).alias(name) if name in numeric else F.col(name)\n        for name in canonical.columns\n    ])\n\n\ndef feature_frame(clean, contract):\n    # Hash the shared cleaned feature vector in fixed contract order, with a policy version prefix.\n    values = F.array(*[F.col(name) for name in contract.feature_names])\n    split_key = F.unhex(F.sha2(F.concat(F.lit(contract.policy["version"] + "\\u0000"), F.to_json(values)), 256))\n    with_key = clean.withColumn("split_feature_key", split_key)\n    return with_key.select(*contract.feature_names, *TARGETS, *FEATURE_CONTEXT)\n\n\ndef exact_groups(frame):\n    return (frame.groupBy("record_fingerprint")\n            .agg(F.count("*").alias("multiplicity"), F.min("source_file").alias("representative_source"))\n            .where(F.col("multiplicity") > 1))\n\n\ndef conflict_groups(frame):\n    return (frame.groupBy("split_feature_key")\n            .agg(F.count("*").alias("rows"), F.count_distinct("original_label").alias("label_count"))\n            .where(F.col("label_count") > 1))\n\n\ndef apply_duplicate_policy(frame, mode):\n    if mode in {"preserve_all", "diagnostic"}:\n        return frame\n    if mode != "exact_representative":\n        raise ValueError(f"Unsupported duplicate policy: {mode}")\n    # Rows tied on source_file and digest are identical visible occurrences in that source.\n    window = Window.partitionBy("record_fingerprint").orderBy("source_file")\n    return frame.withColumn("_representative", F.row_number().over(window)).where("_representative = 1").drop("_representative")\n', 'src/ingest/parquet_pipeline.py': '"""Execute, reconcile and checkpoint the Phase 2 data layers."""\n\nimport argparse\nfrom collections import Counter\nimport csv\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport time\n\nfrom pyspark import StorageLevel\nfrom pyspark.sql import Observation, functions as F\n\nfrom src.ingest.cleaning import apply_duplicate_policy, canonical_frame, clean_frame, conflict_groups, exact_groups, feature_frame\nfrom src.ingest.dataset_audit import atomic_json, count_if, read_inventory, spark_session\nfrom src.ingest.schema_contract import DataContract, SOURCE_COUNTS, reconcile_source, source_records\nfrom src.sql.aggregation_demo import run_aggregation_demo\n\n\nLAYERS = ("raw_aligned", "clean", "feature_ready")\nKINDS = ("nan", "positive_infinity", "negative_infinity", "invalid_negative", "sentinel", "cast_failure", "source_null")\n\n\ndef canonical_metrics():\n    issues = F.col("quality_issues")\n    metrics = [\n        F.count("*").alias("canonical_rows"),\n        count_if(F.col("binary_label") == 0).alias("benign"),\n        count_if(F.col("binary_label") == 1).alias("attack"),\n        count_if(F.col("binary_label").isNull()).alias("unknown_labels"),\n        count_if(F.col("timestamp_parse_status") == "outside_capture_day").alias("timestamp_outlier"),\n        count_if(F.col("timestamp_parse_status") == "unparseable").alias("timestamp_unparseable"),\n        count_if(F.col("timestamp_parse_status") == "missing").alias("timestamp_missing"),\n        count_if(F.col("temporal_eligibility")).alias("temporal_eligible"),\n        count_if(F.size(issues) > 0).alias("quality_affected_rows"),\n        F.coalesce(F.sum(F.size(issues)), F.lit(0)).alias("quality_affected_cells"),\n        count_if(F.size("structural_missing_fields") > 0).alias("structural_missing_rows"),\n        F.coalesce(F.sum(F.size("structural_missing_fields")), F.lit(0)).alias("structural_missing_cells"),\n    ]\n    for reason in KINDS:\n        selected = F.size(F.filter(issues, lambda issue: issue["reason"] == reason))\n        metrics.extend([\n            count_if(selected > 0).alias(reason + "_rows"),\n            F.coalesce(F.sum(selected), F.lit(0)).alias(reason + "_cells"),\n        ])\n    return metrics\n\n\ndef parquet_footers(path):\n    """Verify files and row counts via footer metadata only."""\n    import pyarrow.parquet as parquet\n\n    files = sorted(path.rglob("part-*.parquet"))\n    if not files:\n        raise ValueError(f"No Parquet data files under {path.name}")\n    rows, size = 0, 0\n    codecs = set()\n    for file in files:\n        metadata = parquet.read_metadata(file)\n        rows += metadata.num_rows\n        size += file.stat().st_size\n        for group in range(metadata.num_row_groups):\n            row_group = metadata.row_group(group)\n            codecs.update(row_group.column(i).compression for i in range(row_group.num_columns))\n    if codecs != {"SNAPPY"}:\n        raise ValueError(f"Parquet compression mismatch in {path.name}: {sorted(codecs)}")\n    return {"records": rows, "bytes": size, "files": len(files), "compression": "SNAPPY"}\n\n\ndef write_day(frame, path, number_of_files, expected_rows):\n    started = time.monotonic()\n    (frame.drop("source_day").coalesce(number_of_files).write.mode("overwrite")\n     .option("compression", "snappy").parquet(str(path)))\n    write_s = round(time.monotonic() - started, 3)\n    observed = parquet_footers(path)\n    if observed["records"] != expected_rows:\n        raise ValueError(f"Parquet row-count mismatch at {path.name}")\n    return observed | {"write_seconds": write_s}\n\n\ndef driver_peak_bytes(spark):\n    pid = spark._jvm.java.lang.ProcessHandle.current().pid()\n    status = Path(f"/proc/{pid}/status")\n    if status.is_file():\n        line = next((line for line in status.read_text().splitlines() if line.startswith("VmHWM:")), None)\n        if line is not None:\n            return int(line.split()[1]) * 1024\n    return None\n\n\ndef process_capture(spark, root, stores, contract, source, formats):\n    source_name, day = source["name"], source["capture_day"]\n    path = root / source["path"]\n    expected = source["counts"]\n    start = time.monotonic()\n    variant, frame = source_records(spark, contract, path, day)\n    physical = Observation()\n    measured = frame.observe(\n        physical, F.count("*").alias("physical_lines"),\n        *[count_if(F.col("kind") == name).alias(name) for name in SOURCE_COUNTS],\n    )\n    normalized = Observation()\n    canonical = (canonical_frame(measured, contract, variant, source_name, day, formats)\n                 .observe(normalized, *canonical_metrics()).persist(StorageLevel.DISK_ONLY))\n    target = contract.policy["target_source_bytes_per_part"]\n    parts = max(1, (source["bytes"] + target - 1) // target)\n    try:\n        materialized = canonical.count()\n        parse_seconds = round(time.monotonic() - start, 3)\n        reconciliation = reconcile_source(physical.get, expected)\n        stats = normalized.get\n        if stats["canonical_rows"] != expected["parsed"] or materialized != expected["parsed"]:\n            raise ValueError("Canonical rows failed reconciliation")\n        if stats["unknown_labels"] or stats["benign"] + stats["attack"] != materialized:\n            raise ValueError(f"Unmapped labels in {source_name}: {stats[\'unknown_labels\']}")\n        if stats["timestamp_outlier"] + stats["timestamp_missing"] + stats["timestamp_unparseable"] + stats["temporal_eligible"] != materialized:\n            raise ValueError("Canonical timestamp statuses failed reconciliation")\n        raw_output = write_day(canonical, stores["raw_aligned"] / f"source_day={day}", parts, materialized)\n        clean_output = write_day(clean_frame(canonical, contract), stores["clean"] / f"source_day={day}", parts, materialized)\n    finally:\n        canonical.unpersist(blocking=True)\n    # The third layer is built from the persisted clean data.\n    clean_path = stores["clean"] / f"source_day={day}"\n    feature_source = spark.read.parquet(str(clean_path)).withColumn("source_day", F.lit(day))\n    feature_output = write_day(feature_frame(feature_source, contract), stores["feature_ready"] / f"source_day={day}", parts, materialized)\n    return {\n        "source_file": source_name, "source_day": day, "schema_variant": variant,\n        "source_bytes": source["bytes"], "expected_records": expected["parsed"],\n        "expected_headers": expected["headers"], "read_transform_seconds": parse_seconds,\n        "effective_read_mib_s": round(source["bytes"] / (1024 ** 2) / max(parse_seconds, 0.001), 3),\n        "driver_peak_rss_bytes": driver_peak_bytes(spark),\n        "counts": reconciliation | stats,\n        "layers": dict(zip(LAYERS, (raw_output, clean_output, feature_output))),\n    }\n\n\ndef reconcile_full(summary, expected):\n    counts = summary["totals"]\n    observed = {key: counts[key] for key in ("benign", "attack", "timestamp_outlier")}\n    observed.update(nan=counts["nan_cells"], infinity=counts["positive_infinity_cells"] + counts["negative_infinity_cells"],\n                    invalid_negative=counts["invalid_negative_cells"])\n    if any(observed[key] != expected[key] for key in observed):\n        raise ValueError(f"Phase 1 totals disagree: {observed}")\n    if counts["canonical_rows"] != counts["parsed"] or any(counts["canonical_rows"] != summary["layers"][name]["records"] for name in LAYERS):\n        raise ValueError("Layer totals do not reconcile")\n\n\ndef layer_summary(files, data_root):\n    output = {}\n    for name in LAYERS:\n        metrics = [row["layers"][name] for row in files]\n        output[name] = {\n            "path": (data_root / name).as_posix(), "partition_column": "source_day",\n            "partitions": len(metrics), "format": "parquet", "compression": "SNAPPY",\n            **{key: sum(item[key] for item in metrics) for key in ("records", "bytes", "files")},\n            "write_seconds": round(sum(item["write_seconds"] for item in metrics), 3),\n        }\n    return output\n\n\ndef write_csv(path, rows, fields):\n    with path.open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fields, lineterminator="\\n")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef write_evidence(evidence, summary, policy):\n    atomic_json(evidence / "validation_summary.json", summary)\n    atomic_json(evidence / "parquet_layers.json", {\n        key: summary[key] for key in ("status", "dataset", "policy_version", "layers", "duplicate_policy", "duplicate_diagnostics")\n        if key in summary\n    })\n    captured = summary["completed_files"]\n    counts = [{\n        "source_file": item["source_file"], "source_day": item["source_day"],\n        "expected_records": item["expected_records"], "observed_records": item["counts"]["parsed"],\n        "physical_lines": item["counts"]["physical_lines"], "expected_headers": item["expected_headers"],\n        "headers": item["counts"]["headers"], "repeated_headers": item["counts"]["repeated_headers"],\n        "canonical": item["counts"]["canonical_rows"], "clean": item["layers"]["clean"]["records"],\n        "feature_ready": item["layers"]["feature_ready"]["records"], "benign": item["counts"]["benign"],\n        "attack": item["counts"]["attack"], "timestamp_outlier": item["counts"]["timestamp_outlier"],\n    } for item in captured]\n    timing = [{\n        "source_file": item["source_file"], "source_bytes": item["source_bytes"],\n        "read_transform_seconds": item["read_transform_seconds"], "effective_read_mib_s": item["effective_read_mib_s"],\n        "driver_peak_rss_bytes": item["driver_peak_rss_bytes"],\n        **{name + "_write_seconds": item["layers"][name]["write_seconds"] for name in LAYERS},\n    } for item in captured]\n    quality = [{\n        "source_file": item["source_file"], "reason": reason,\n        "affected_rows": item["counts"][reason + "_rows"], "affected_cells": item["counts"][reason + "_cells"],\n        "action": policy["numeric_actions"].get(reason, "structural_null" if reason == "structural_missing" else "all_quality_issues"),\n    } for item in captured for reason in (*KINDS, "structural_missing", "quality_affected")]\n    if counts:\n        write_csv(evidence / "phase2_count_reconciliation.csv", counts, list(counts[0]))\n        write_csv(evidence / "phase2_performance.csv", timing, list(timing[0]))\n        write_csv(evidence / "phase2_quality_actions.csv", quality, list(quality[0]))\n\n\ndef run_duplicate_mode(spark, stores, policy, data_root, expected_excess):\n    mode = policy["duplicate_policy"]\n    if mode == "preserve_all":\n        return {"records_removed_primary": 0}\n    if mode not in {"diagnostic", "exact_representative"}:\n        raise ValueError(f"Unknown duplicate policy {mode}")\n    started = time.monotonic()\n    folder = data_root / "diagnostics"\n    folder.mkdir(exist_ok=True)\n    aligned = spark.read.parquet(str(stores["raw_aligned"])).select("record_fingerprint", "source_file")\n    keys = folder / "exact_groups"\n    exact_groups(aligned).write.mode("overwrite").option("compression", "snappy").parquet(str(keys))\n    exact = spark.read.parquet(str(keys)).agg(F.count("*").alias("groups"),\n                                              F.coalesce(F.sum(F.col("multiplicity") - 1), F.lit(0)).alias("excess")).first().asDict()\n    if expected_excess is not None and exact["excess"] != expected_excess:\n        raise ValueError("Exact raw-line duplicate excess differs from Phase 1")\n    features = spark.read.parquet(str(stores["feature_ready"])).select("split_feature_key", "original_label")\n    conflicts_path = folder / "label_conflicts"\n    conflict_groups(features).write.mode("overwrite").option("compression", "snappy").parquet(str(conflicts_path))\n    conflicts = spark.read.parquet(str(conflicts_path)).agg(F.count("*").alias("groups"),\n        F.coalesce(F.sum("rows"), F.lit(0)).alias("participating_rows")).first().asDict()\n    result = {"records_removed_primary": 0, "exact": exact, "cleaned_feature_conflicts": conflicts,\n              "diagnostics_path": folder.as_posix()}\n    if mode == "exact_representative":\n        clean = spark.read.parquet(str(stores["clean"]))\n        target = data_root / "exact_representatives"\n        (apply_duplicate_policy(clean, mode).write.mode("overwrite").option("compression", "snappy")\n         .partitionBy("source_day").parquet(str(target)))\n        measured = parquet_footers(target)\n        if measured["records"] != sum(parquet_footers(p)["records"] for p in stores["clean"].glob("source_day=*")) - exact["excess"]:\n            raise ValueError("Representative output failed multiplicity reconciliation")\n        result["representatives"] = measured | {"path": target.as_posix()}\n    return result | {"seconds": round(time.monotonic() - started, 3)}\n\n\ndef resource_limits(spark, policy, path):\n    metrics = {"spark_version": spark.version, **policy["spark"], "default_parallelism": spark.sparkContext.defaultParallelism,\n               "cpu_affinity": len(os.sched_getaffinity(0)), "scratch_output_free_bytes_start": shutil.disk_usage(path).free}\n    for name in ("memory.max", "cpu.max"):\n        resource = Path("/sys/fs/cgroup") / name\n        if resource.exists():\n            metrics[name] = resource.read_text().strip()\n    return metrics\n\n\ndef run_pipeline(contract, source_config, phase1, root, output, scratch, only_days=None):\n    root, output, scratch = map(Path, (root, output, scratch))\n    if not root.is_dir() or any(p.resolve().is_relative_to(root.resolve()) for p in (output, scratch)):\n        raise ValueError("The verified mount and writable outputs must be separate")\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    actual = read_inventory(root, source_config)\n    audited = {row["path"]: row for row in phase1["files"]}\n    if len(actual) != len(audited) or any(\n        row["path"] not in audited or row["bytes"] != audited[row["path"]]["bytes"]\n        or row["access"] != "readable" or row["unexpected"] or row["duplicate_basename"] for row in actual\n    ):\n        raise ValueError("Mounted inventory differs from Phase 1")\n    policy = contract.policy\n    data_root = output / policy["version"]\n    stores = {name: data_root / name for name in LAYERS}\n    evidence = output / "evidence"\n    evidence.mkdir(exist_ok=True)\n    spark = spark_session({"spark": policy["spark"]}, scratch)\n    spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")\n    spark.sparkContext.setLogLevel("WARN")\n    summary = {\n        "status": "running", "dataset": phase1["dataset"], "policy_version": policy["version"],\n        "duplicate_policy": policy["duplicate_policy"], "expected_files": len(audited),\n        "resources": resource_limits(spark, policy, output), "completed_files": [], "failures": [],\n        "phase1_conflict_groups": 22129,\n    }\n    started = time.monotonic()\n    totals = Counter()\n    try:\n        sources = sorted(audited.values(), key=lambda file: file["bytes"])\n        for source in sources:\n            day = source["capture_day"]\n            if only_days is not None and day not in only_days:\n                continue\n            try:\n                marker = evidence / f"capture-{day}.json"\n                if marker.is_file():\n                    result = json.loads(marker.read_text())\n                    for name in LAYERS:\n                        metrics = parquet_footers(stores[name] / f"source_day={day}")\n                        if any(result["layers"][name][key] != value for key, value in metrics.items()):\n                            raise ValueError("Existing checkpoint no longer matches its data files")\n                else:\n                    print(f"PHASE2 start {day} {source[\'bytes\']} bytes", flush=True)\n                    result = process_capture(spark, root, stores, contract, source, source_config["timestamp_formats"])\n                    atomic_json(marker, result)\n                summary["completed_files"].append(result)\n                totals.update(result["counts"])\n                print(f"PHASE2 completed {day} {result[\'counts\'][\'canonical_rows\']} rows "\n                      f"in {result[\'read_transform_seconds\']} s read and transform", flush=True)\n            except Exception as error:\n                summary["failures"].append({"source_day": day, "stage": "capture", "type": type(error).__name__})\n                print(f"PHASE2 failed {day}: {type(error).__name__}: {str(error)[:1000]}", flush=True)\n            summary.update(totals=dict(totals), layers=layer_summary(summary["completed_files"], data_root))\n            write_evidence(evidence, summary, policy)\n        full = len(summary["completed_files"]) == len(sources)\n        if full:\n            reconcile_full(summary, policy["expected_full"])\n            stored = stores["feature_ready"] / f"source_day={sources[0][\'capture_day\']}"\n            summary["aggregation_demo"] = run_aggregation_demo(spark.read.parquet(str(stored)))\n            summary["duplicate_diagnostics"] = run_duplicate_mode(\n                spark, stores, policy, data_root, policy["expected_full"]["duplicate_excess"])\n        summary["status"] = "validated" if full else "partial"\n    except Exception as error:\n        summary["status"] = "partial"\n        summary["failures"].append({"stage": "global_validation", "type": type(error).__name__})\n        print(f"PHASE2 validation failed: {type(error).__name__}: {str(error)[:1000]}", flush=True)\n    finally:\n        summary["elapsed_seconds"] = round(time.monotonic() - started, 3)\n        summary["resources"]["driver_peak_rss_bytes"] = driver_peak_bytes(spark)\n        write_evidence(evidence, summary, policy)\n        print(f"PHASE2 final {summary[\'status\']} in {summary[\'elapsed_seconds\']} seconds", flush=True)\n        # Persist compact evidence in logs as well as the notebook output.\n        print("PHASE2_EVIDENCE " + json.dumps(summary, separators=(",", ":")), flush=True)\n        spark.stop()\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--scratch", type=Path, required=True)\n    parser.add_argument("--day", action="append")\n    args = parser.parse_args()\n    contract = DataContract.load("configs/cleaning.yaml")\n    load = lambda path: json.loads(Path(path).read_text())\n    summary = run_pipeline(contract, load("configs/dataset.yaml"), load("manifests/dataset_inventory.json"),\n                           args.root, args.output, args.scratch, args.day)\n    raise SystemExit(0 if summary["status"] == "validated" else 1)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/sql/aggregation_demo.py': '"""Compare equivalent SQL, DataFrame and RDD aggregations on a bounded sample."""\n\nfrom pyspark.sql import functions as F\n\n\ndef run_aggregation_demo(features):\n    sample = (features.select("protocol", "binary_label", "record_fingerprint")\n              .where(F.pmod(F.xxhash64("record_fingerprint"), F.lit(1024)) == 0)\n              .select("protocol", "binary_label").cache())\n    sample.createOrReplaceTempView("phase2_demo_sample")\n    try:\n        sql = features.sparkSession.sql(\n            "SELECT protocol, count(*) AS records, sum(binary_label) AS attacks "\n            "FROM phase2_demo_sample GROUP BY protocol")\n        dataframe = sample.groupBy("protocol").agg(F.count("*").alias("records"), F.sum("binary_label").alias("attacks"))\n        rdd = (sample.rdd.map(lambda row: (row.protocol, (1, row.binary_label)))\n               .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])))\n        result = {row.protocol: (row.records, row.attacks) for row in sql.collect()}\n        if result != {row.protocol: (row.records, row.attacks) for row in dataframe.collect()} or result != dict(rdd.collect()):\n            raise ValueError("SQL, DataFrame and RDD aggregation disagree")\n        if not result:\n            raise ValueError("The demonstration sample is empty")\n        return {\n            "sample_rule": "smallest capture by source bytes; pmod(xxhash64(record_fingerprint), 1024) == 0",\n            "rows": sum(count for count, _ in result.values()),\n            "groups": [{"protocol": key, "records": value[0], "attacks": value[1]}\n                       for key, value in sorted(result.items())],\n            "sql_plan": sql._jdf.queryExecution().executedPlan().toString(),\n            "dataframe_plan": dataframe._jdf.queryExecution().executedPlan().toString(),\n            "rdd_dag": rdd.toDebugString().decode("utf-8"),\n        }\n    finally:\n        features.sparkSession.catalog.dropTempView("phase2_demo_sample")\n        sample.unpersist()\n', 'src/sql/traffic_analytics.py': '"""Validated input and reusable SQL aggregations for Phase 3."""\n\nfrom collections import Counter\nimport csv\nfrom hashlib import sha256\nimport json\nimport math\nfrom pathlib import Path\nimport re\nimport time\n\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.dataset_audit import atomic_json\nfrom src.ingest.parquet_pipeline import parquet_footers\nfrom src.ingest.schema_contract import FEATURE_CONTEXT, TARGETS\n\n\ndef identifier(name):\n    if not re.fullmatch(r"[a-z][a-z0-9_]*", name):\n        raise ValueError(f"Invalid configured SQL identifier {name!r}")\n    return name\n\n\ndef locate_producer(input_root, expected_sha):\n    relative = "mmds_phase2/evidence/validation_summary.json"\n    root = Path(input_root)\n    candidates = set(root.rglob(relative))\n    for depth in (1, 2, 3, 4):\n        candidates.update(root.glob("*/" * depth + relative))\n    matched = {path.resolve(): path for path in sorted(candidates)\n               if sha256(path.read_bytes()).hexdigest() == expected_sha}\n    if len(matched) != 1:\n        raise ValueError("Expected one mounted input matching the Phase 2 receipt")\n    return next(iter(matched.values()))\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef label_grouping(frame):\n    return frame.groupBy("source_day", *TARGETS, "temporal_eligibility").count()\n\n\ndef compact(frame, maximum):\n    rows = frame.limit(maximum + 1).collect()\n    require(len(rows) <= maximum, "Aggregate exceeds compact output limit")\n    def scalar(value):\n        if isinstance(value, float):\n            return value if math.isfinite(value) else None\n        return value\n\n    return [{k: scalar(v) for k, v in row.asDict().items()} for row in rows]\n\n\ndef validate_input(spark, receipt, manifest, policy):\n    summary = json.loads(receipt.read_text())\n    expected_sha = manifest["independent_mount"]["summary_sha256"]\n    require(sha256(receipt.read_bytes()).hexdigest() == expected_sha, "Phase 2 receipt checksum differs")\n    require(summary["status"] == "validated" and summary["dataset"] == manifest["dataset"], "Wrong producer dataset")\n    require(summary["policy_version"] == manifest["policy_version"] == policy["version"], "Wrong cleaning policy")\n    require(summary["totals"] == manifest["totals"], "Wrong producer totals")\n    root = receipt.parent.parent / summary["policy_version"]\n    frames = {}\n    evidence = {}\n    for name in ("feature_ready", "clean", "raw_aligned"):\n        directory = root / name\n        actual = parquet_footers(directory)\n        expected = manifest["layers"][name]\n        days = sorted(path.name.split("=", 1)[1] for path in directory.glob("source_day=*") if path.is_dir())\n        require(all(expected[key] == value for key, value in actual.items()), f"{name} is unreadable or differs from manifest")\n        require(len(days) == expected["partitions"] == 10, f"{name} partition count differs")\n        require(days == sorted(row["source_day"] for row in summary["completed_files"]), f"{name} day list differs")\n        frames[name] = spark.read.parquet(str(directory))\n        evidence[name] = actual | {"path": str(directory), "partitions": len(days)}\n    features, clean = frames["feature_ready"], frames["clean"]\n    require(set(features.columns) == set(manifest["feature_columns"]) | set(FEATURE_CONTEXT) | set(TARGETS), "Feature schema differs")\n    for frame in (features, clean):\n        types = dict(frame.dtypes)\n        require(all(types.get(name) == "double" for name in manifest["feature_columns"]), "Feature types differ")\n        require(types.get("source_day") == "string" and types.get("binary_label") == "tinyint", "Target or partition type differs")\n        require(types.get("record_fingerprint") == "binary" and types.get("temporal_eligibility") == "boolean", "Provenance types differ")\n    require(dict(clean.dtypes).get("parsed_event_time") == "timestamp_ntz", "Wall-clock timestamp is unavailable")\n    require({row[0] for row in clean.select("cleaning_policy_version").distinct().collect()} == {policy["version"]}, "Stored cleaning policy differs")\n    clean_groups = compact(label_grouping(clean).orderBy("source_day", "original_label", "temporal_eligibility"), 500)\n    feature_groups = compact(label_grouping(features).orderBy("source_day", "original_label", "temporal_eligibility"), 500)\n    require(clean_groups == feature_groups, "Layer targets or temporal flags differ")\n    expected_totals = summary["totals"]\n    labels, day_counts = Counter(), Counter()\n    ineligible = 0\n    for row in clean_groups:\n        label = policy["label_mapping"].get(row["original_label"])\n        require(label is not None and (row["normalized_label"], row["binary_label"], row["attack_family"]) ==\n                (label["normalized"], label["binary"], label["family"]), "Observed label mapping differs")\n        require(row["temporal_eligibility"] in (True, False), "Null temporal flag")\n        labels[row["binary_label"]] += row["count"]\n        day_counts[row["source_day"]] += row["count"]\n        if not row["temporal_eligibility"]:\n            ineligible += row["count"]\n    require(dict(labels) == {0: expected_totals["benign"], 1: expected_totals["attack"]}, "Binary counts differ")\n    require(sum(labels.values()) == expected_totals["canonical_rows"] == 16232943 and ineligible == 14, "Row or temporal counts differ")\n    require(dict(day_counts) == {row["source_day"]: row["counts"]["canonical_rows"] for row in summary["completed_files"]}, "Per-day counts differ")\n    bad_time = clean.where(\n        ~F.col("temporal_eligibility").eqNullSafe(\n            F.col("parsed_event_time").isNotNull() & (F.to_date("parsed_event_time") == F.col("source_day").cast("date"))))\n    require(bad_time.limit(1).count() == 0, "Temporal eligibility disagrees with stored event time")\n    return frames, root, {\n        "status": "validated", "dataset": summary["dataset"], "phase2_receipt_sha256": expected_sha,\n        "policy_version": summary["policy_version"], "producer": manifest["producer"], "layers": evidence,\n        "records": sum(labels.values()), "benign": labels[0], "attack": labels[1], "temporal_ineligible": ineligible,\n        "source_days": sorted(day_counts), "features": manifest["feature_columns"],\n    }\n\n\ndef finite(column):\n    name = identifier(column)\n    return f"CASE WHEN {name} IS NOT NULL AND NOT isnan({name}) AND abs({name}) < CAST(\'Infinity\' AS DOUBLE) THEN {name} END"\n\n\ndef register_traffic(clean, metrics):\n    """Project only the columns needed by exploratory queries."""\n    selected = set(metrics) - {"total_packets", "total_bytes"}\n    columns = ["source_day", *TARGETS, "temporal_eligibility", "parsed_event_time",\n               *sorted(selected | {"protocol", "dst_port", "fin_flag_cnt"})]\n    projected = (clean.select(*columns, "quality_issues")\n                 .withColumn("quality_cells", F.size("quality_issues"))\n                 .withColumn("invalid_cells", F.size(F.filter("quality_issues", lambda issue: issue["reason"] != "sentinel")))\n                 .drop("quality_issues"))\n    return (\n        projected.withColumn("population", F.when(F.col("binary_label") == 0, "benign").otherwise("attack"))\n        .withColumn("total_packets", F.col("tot_fwd_pkts") + F.col("tot_bwd_pkts"))\n        .withColumn("total_bytes", F.col("totlen_fwd_pkts") + F.col("totlen_bwd_pkts"))\n    )\n\n\nDAILY_SQL = """SELECT source_day, count(*) AS records, count_if(binary_label = 0) AS benign,\n count_if(binary_label = 1) AS attacks, avg(CAST(binary_label AS DOUBLE)) AS attack_share,\n sum(total_packets) AS packet_count, sum(total_bytes) AS byte_count\n FROM traffic GROUP BY source_day ORDER BY source_day"""\n\nTOTAL_SQL = """SELECT population, count(*) AS records, sum(total_packets) AS packet_count,\n sum(total_bytes) AS byte_count, count(total_packets) AS packet_observations,\n count(total_bytes) AS byte_observations FROM traffic GROUP BY population ORDER BY population"""\n\nFAMILY_SQL = """SELECT attack_family, count(*) AS records, count(*) / sum(count(*)) OVER () AS share_of_attacks,\n count(DISTINCT source_day) AS capture_days,\n count_if(quality_cells > 0) AS quality_affected_rows, avg(CAST(quality_cells > 0 AS DOUBLE)) AS quality_affected_share,\n count_if(invalid_cells > 0) AS invalid_affected_rows, avg(CAST(invalid_cells > 0 AS DOUBLE)) AS invalid_affected_share\n FROM traffic WHERE binary_label = 1 GROUP BY attack_family ORDER BY records DESC, attack_family"""\n\nLABEL_SQL = """SELECT attack_family, original_label, count(*) AS records, count(DISTINCT source_day) AS capture_days\n FROM traffic GROUP BY attack_family, original_label ORDER BY attack_family, original_label"""\n\nPROTOCOL_SQL = """SELECT population, attack_family, protocol, count(*) AS records\n FROM traffic GROUP BY population, attack_family, protocol ORDER BY population, attack_family, protocol"""\n\nPORT_SQL = """WITH grouped AS (\n SELECT population, coalesce(attack_family, \'All families\') AS attack_family,\n CASE WHEN grouping(attack_family) = 1 THEN \'population\' ELSE \'family\' END AS scope,\n dst_port, count(*) AS records FROM traffic\n GROUP BY GROUPING SETS ((population, attack_family, dst_port), (population, dst_port))),\n ranked AS (SELECT *, row_number() OVER (PARTITION BY scope, population, attack_family ORDER BY records DESC, dst_port) AS rank,\n sum(records) OVER (PARTITION BY scope, population, attack_family) AS population_records FROM grouped)\n SELECT * FROM ranked WHERE rank <= {limit} ORDER BY scope, population, attack_family, rank"""\n\ndef feature_statistics_sql(features, group, accuracy):\n    group = identifier(group)\n    terms = []\n    for feature in features:\n        x = finite(feature)\n        terms.append(\n            f"named_struct(\'feature\', \'{feature}\', \'valid\', count({x}), \'population_count\', count(*), "\n            f"\'mean\', avg({x}), \'stddev\', stddev_samp({x}), \'min\', min({x}), \'max\', max({x}), "\n            f"\'quantiles\', percentile_approx({x}, array(0.1D, 0.5D, 0.9D, 0.99D), {int(accuracy)}))"\n        )\n    return (f"SELECT {group}, m.* FROM (SELECT {group}, explode(array({\', \'.join(terms)})) AS m "\n            f"FROM traffic GROUP BY {group}) ORDER BY {group}, m.feature")\n\n\ndef missing_sql(features):\n    terms = [\n        f"named_struct(\'feature\', \'{name}\', \'missing\', count(*) - count({finite(name)}), \'records\', count(*))"\n        for name in features\n    ]\n    return f"SELECT m.* FROM (SELECT explode(array({\', \'.join(terms)})) AS m FROM features) ORDER BY m.feature"\n\n\ndef correlations_sql(pairs):\n    terms = []\n    for x, y in pairs:\n        a, b = finite(x), finite(y)\n        terms.append(\n            f"named_struct(\'x\', \'{x}\', \'y\', \'{y}\', \'pairs\', count_if(({a}) IS NOT NULL AND ({b}) IS NOT NULL), "\n            f"\'pearson\', corr({a}, {b}))"\n        )\n    return (f"SELECT population, m.* FROM (SELECT population, explode(array({\', \'.join(terms)})) AS m "\n            "FROM traffic GROUP BY population) ORDER BY population, m.x, m.y")\n\n\ndef write_rows(destination, rows):\n    if not rows:\n        raise ValueError(f"Unexpected empty result {destination.name}")\n    stage = destination.with_suffix(".tmp")\n    with stage.open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]), lineterminator="\\n")\n        writer.writeheader()\n        writer.writerows({key: json.dumps(value, separators=(",", ":")) if isinstance(value, (list, dict)) else value\n                          for key, value in row.items()} for row in rows)\n    stage.replace(destination)\n\n\ndef timed_sql(spark, query, limit, name, output, performance):\n    started = time.monotonic()\n    rows = compact(spark.sql(query), limit)\n    seconds = round(time.monotonic() - started, 6)\n    write_rows(output / f"phase3_{name}.csv", rows)\n    performance.append({"workload": name, "api": "SQL", "iteration": 1, "seconds": seconds, "result_rows": len(rows)})\n    print(f"PHASE3 query {name}: {len(rows)} result rows in {seconds:.3f} s", flush=True)\n    return rows\n\n\ndef write_checkpoint(output, validation, completed, performance):\n    atomic_json(output / "phase3_validation.json", validation | {"completed_queries": list(completed)})\n    if performance:\n        write_rows(output / "phase3_query_performance.csv", performance)\n', 'src/sql/temporal_analytics.py': '"""Capture-day and eligible local wall-clock summaries."""\n\nFAMILY_DAY_SQL = """SELECT d.source_day, f.attack_family, coalesce(o.records, 0) AS records\n FROM (SELECT DISTINCT source_day FROM traffic) d\n CROSS JOIN (SELECT DISTINCT attack_family FROM traffic WHERE binary_label = 1) f\n LEFT JOIN (SELECT source_day, attack_family, count(*) AS records FROM traffic WHERE binary_label = 1\n GROUP BY source_day, attack_family) o\n ON d.source_day = o.source_day AND f.attack_family = o.attack_family\n ORDER BY d.source_day, f.attack_family"""\n\nHOURLY_SQL = """SELECT source_day, hour(parsed_event_time) AS local_hour, count(*) AS records,\n count_if(binary_label = 0) AS benign, count_if(binary_label = 1) AS attacks,\n avg(CAST(binary_label AS DOUBLE)) AS attack_share,\n avg(flow_duration) AS mean_duration_us, avg(total_packets) AS mean_packets, avg(total_bytes) AS mean_bytes,\n count(flow_duration) AS duration_observations, count(total_packets) AS packet_observations,\n count(total_bytes) AS byte_observations\n FROM traffic WHERE temporal_eligibility\n GROUP BY source_day, hour(parsed_event_time) ORDER BY source_day, local_hour"""\n\nWINDOW_SQL = """SELECT source_day, population, floor(hour(parsed_event_time) / 3) * 3 AS window_start_hour,\n count(*) AS records, count(flow_duration) AS duration_observations, count(total_packets) AS packet_observations,\n count(total_bytes) AS byte_observations, percentile_approx(flow_duration, 0.5D, {accuracy}) AS median_duration_us,\n percentile_approx(total_packets, 0.5D, {accuracy}) AS median_packets,\n percentile_approx(total_bytes, 0.5D, {accuracy}) AS median_bytes\n FROM traffic WHERE temporal_eligibility\n GROUP BY source_day, population, floor(hour(parsed_event_time) / 3) ORDER BY source_day, window_start_hour, population"""\n', 'src/sql/attack_patterns.py': '"""Exploratory pattern summaries for labeled historical traffic."""\n\nfrom src.sql.traffic_analytics import finite\n\n\ndef benign_thresholds_sql(accuracy):\n    return f"""SELECT percentile_approx(flow_pkts_s, 0.99D, {int(accuracy)}) AS packet_rate_p99,\n percentile_approx(total_packets, 0.99D, {int(accuracy)}) AS packet_count_p99,\n percentile_approx(total_bytes, 0.99D, {int(accuracy)}) AS byte_count_p99,\n percentile_approx(flow_duration, 0.1D, {int(accuracy)}) AS duration_p10,\n percentile_approx(flow_iat_mean, 0.99D, {int(accuracy)}) AS iat_p99\n FROM traffic WHERE population = \'benign\'"""\n\n\ndef patterns_sql(thresholds):\n    rules = [\n        ("high_packet_rate", "flow_pkts_s IS NOT NULL", f"flow_pkts_s > {float(thresholds[\'packet_rate_p99\'])!r}"),\n        ("large_packet_count", "total_packets IS NOT NULL", f"total_packets > {float(thresholds[\'packet_count_p99\'])!r}"),\n        ("short_high_bytes", "flow_duration IS NOT NULL AND total_bytes IS NOT NULL",\n         f"flow_duration <= {float(thresholds[\'duration_p10\'])!r} AND total_bytes > {float(thresholds[\'byte_count_p99\'])!r}"),\n        ("high_mean_iat", "flow_iat_mean IS NOT NULL", f"flow_iat_mean > {float(thresholds[\'iat_p99\'])!r}"),\n    ]\n    stats = [f"named_struct(\'pattern\', \'{name}\', \'eligible\', count_if({valid}), \'matches\', count_if({valid} AND {condition}))"\n             for name, valid, condition in rules]\n    return (f"SELECT population, attack_family, m.* FROM (SELECT population, attack_family, explode(array({\', \'.join(stats)})) AS m "\n            "FROM traffic GROUP BY population, attack_family) ORDER BY population, attack_family, m.pattern")\n\n\nFLAGS_SQL = """SELECT population, attack_family, syn_flag_cnt > 0 AS syn, ack_flag_cnt > 0 AS ack,\n fin_flag_cnt > 0 AS fin, rst_flag_cnt > 0 AS rst, count(*) AS records\n FROM traffic WHERE protocol = 6\n GROUP BY population, attack_family, syn, ack, fin, rst\n ORDER BY population, attack_family, records DESC, syn, ack, fin, rst"""\n\n\ndef sensitivity_sql(layer):\n    if layer not in {"raw_aligned", "clean"}:\n        raise ValueError("Unknown sensitivity view")\n    fields = ("flow_byts_s", "flow_pkts_s", "init_fwd_win_byts", "init_bwd_win_byts", "flow_duration")\n    terms = []\n    for name in fields:\n        x = finite(name)\n        terms.append(\n            f"named_struct(\'feature\', \'{name}\', \'records\', count(*), \'nulls\', count_if({name} IS NULL), "\n            f"\'nan\', count_if(isnan({name})), \'infinities\', count_if(abs({name}) = CAST(\'Infinity\' AS DOUBLE)), "\n            f"\'minus_one\', count_if({name} = -1D), \'finite_values\', count({x}), \'finite_mean\', avg({x}))"\n        )\n    return (f"SELECT \'{layer}\' AS layer, binary_label, m.* FROM (SELECT binary_label, explode(array({\', \'.join(terms)})) AS m "\n            f"FROM {layer} GROUP BY binary_label) ORDER BY binary_label, m.feature")\n', 'src/sql/query_benchmarks.py': '"""Controlled equivalent SQL and DataFrame workloads."""\n\nimport time\n\nfrom pyspark.sql import functions as F\n\nfrom src.sql.traffic_analytics import compact, require\n\n\ndef compare_workload(frame, repeats):\n    view = "phase3_benchmark"\n    frame.createOrReplaceTempView(view)\n    spark = frame.sparkSession\n    makers = {\n        "SQL": lambda: spark.sql(\n            f"SELECT source_day, binary_label, count(*) AS records FROM {view} "\n            "GROUP BY source_day, binary_label ORDER BY source_day, binary_label"),\n        "DataFrame": lambda: frame.groupBy("source_day", "binary_label").agg(F.count("*").alias("records"))\n        .orderBy("source_day", "binary_label"),\n    }\n    rows, timings, plans = {}, [], {}\n    try:\n        for api, make in makers.items():\n            rows[api] = compact(make(), 40)\n        require(rows["SQL"] == rows["DataFrame"], "Warm-up benchmark mismatch")\n        for i in range(repeats):\n            order = tuple(makers) if i % 2 == 0 else tuple(reversed(makers))\n            for api in order:\n                started = time.monotonic()\n                query = makers[api]()\n                actual = compact(query, 40)\n                elapsed = round(time.monotonic() - started, 6)\n                require(actual == rows["SQL"], f"Benchmark result mismatch: {api}")\n                timings.append({"workload": "day_label_equivalence", "api": api, "iteration": i + 1,\n                                "seconds": elapsed, "result_rows": len(actual)})\n                if api not in plans:\n                    plans[api] = query._jdf.queryExecution().executedPlan().toString()\n    finally:\n        spark.catalog.dropTempView(view)\n    return timings, plans, rows["SQL"]\n', 'src/sql/phase3_pipeline.py': '"""Run the Phase 3 notebook stages against the verified Phase 2 output."""\n\nimport json\nfrom pathlib import Path\nimport time\n\nfrom pyspark import StorageLevel\n\nfrom src.ingest.dataset_audit import atomic_json, spark_session\nfrom src.ingest.parquet_pipeline import driver_peak_bytes, resource_limits\nfrom src.sql.aggregation_demo import run_aggregation_demo\nfrom src.sql.attack_patterns import FLAGS_SQL, benign_thresholds_sql, patterns_sql, sensitivity_sql\nfrom src.sql.query_benchmarks import compare_workload\nfrom src.sql.temporal_analytics import FAMILY_DAY_SQL, HOURLY_SQL, WINDOW_SQL\nfrom src.sql.traffic_analytics import (\n    DAILY_SQL, FAMILY_SQL, LABEL_SQL, PORT_SQL, PROTOCOL_SQL, TOTAL_SQL, correlations_sql,\n    feature_statistics_sql, locate_producer, missing_sql, register_traffic, require, timed_sql,\n    validate_input, write_checkpoint, write_rows,\n)\nfrom src.visualization.eda_charts import draw, load_inputs\n\n\ndef read(path):\n    return json.loads(path.read_text())\n\n\ndef verify_aggregates(data, expected):\n    total = expected["records"]\n    require(sum(row["records"] for row in data["daily_traffic"]) == total, "Capture-day counts do not reconcile")\n    require({row["population"]: row["records"] for row in data["traffic_totals"]} ==\n            {"benign": expected["benign"], "attack": expected["attack"]}, "Binary counts do not reconcile")\n    require(sum(row["records"] for row in data["attack_families"]) == expected["attack"], "Attack-family counts do not reconcile")\n    require(sum(row["records"] for row in data["original_labels"]) == total, "Original-label counts do not reconcile")\n    require(sum(row["records"] for row in data["family_by_day"]) == expected["attack"], "Family-day counts do not reconcile")\n    eligible = total - expected["temporal_ineligible"]\n    require(sum(row["records"] for row in data["hourly_traffic"]) == eligible, "Hourly counts do not reconcile")\n    require(sum(row["records"] for row in data["time_windows"]) == eligible, "Time-window counts do not reconcile")\n\n\ndef run(project, input_root, output, scratch):\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    config = read(project / "configs/analytics.yaml")\n    manifest = read(project / "manifests/parquet_layers.json")\n    policy = read(project / "configs/cleaning.yaml")\n    spark = spark_session(config, scratch)\n    spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")\n    spark.sparkContext.setLogLevel("WARN")\n    started = time.monotonic()\n    view = None\n    timings, data = [], {}\n    try:\n        receipt = locate_producer(input_root, manifest["independent_mount"]["summary_sha256"])\n        frames, root, validation = validate_input(spark, receipt, manifest, policy)\n        validation.update(status="running", records_ready=True, validation_seconds=round(time.monotonic() - started, 3),\n                          resources=resource_limits(spark, config, output))\n        write_checkpoint(output, validation, [], [])\n        print(f"PHASE3 validated input: {validation[\'records\']} rows, {len(validation[\'source_days\'])} days", flush=True)\n        clean, features = frames["clean"], frames["feature_ready"]\n        for name, frame in frames.items():\n            frame.createOrReplaceTempView(name)\n        features.createOrReplaceTempView("features")\n        view = register_traffic(clean, config["flow_statistics"]).persist(StorageLevel.DISK_ONLY)\n        view.createOrReplaceTempView("traffic")\n        settings = config["quantile_accuracy"]\n        metrics = config["flow_statistics"]\n        queries = [\n            ("traffic_totals", TOTAL_SQL), ("daily_traffic", DAILY_SQL),\n            ("attack_families", FAMILY_SQL), ("original_labels", LABEL_SQL),\n            ("protocols", PROTOCOL_SQL), ("destination_ports", PORT_SQL.format(limit=config["top_ports"])),\n            ("family_by_day", FAMILY_DAY_SQL), ("hourly_traffic", HOURLY_SQL),\n            ("time_windows", WINDOW_SQL.format(accuracy=settings)),\n            ("feature_statistics", feature_statistics_sql(metrics, "population", settings)),\n            ("family_feature_statistics", feature_statistics_sql(metrics, "attack_family", settings)),\n            ("feature_missingness", missing_sql(manifest["feature_columns"])),\n            ("correlations", correlations_sql(config["correlations"])), ("tcp_flags", FLAGS_SQL),\n            ("raw_sensitivity", sensitivity_sql("raw_aligned")), ("clean_sensitivity", sensitivity_sql("clean")),\n            ("reference_thresholds", benign_thresholds_sql(settings)),\n        ]\n\n        def execute(name, query):\n            data[name] = timed_sql(spark, query, config["max_compact_rows"], name, output, timings)\n            write_checkpoint(output, validation, data, timings)\n\n        for name, query in queries:\n            execute(name, query)\n        execute("candidate_patterns", patterns_sql(data["reference_thresholds"][0]))\n        for table in ("exact_groups", "label_conflicts"):\n            spark.read.parquet(str(root / "diagnostics" / table)).createOrReplaceTempView(table)\n        execute("duplicate_quality", """SELECT\n (SELECT count(*) FROM exact_groups) AS exact_groups,\n (SELECT sum(multiplicity - 1) FROM exact_groups) AS excess_copies,\n (SELECT sum(multiplicity) FROM exact_groups) AS rows_in_exact_groups,\n (SELECT count(*) FROM label_conflicts) AS feature_label_conflict_groups,\n (SELECT sum(rows) FROM label_conflicts) AS rows_in_feature_label_conflicts,\n (SELECT count(*) FROM traffic WHERE NOT temporal_eligibility) AS temporal_ineligible""")\n        quality = data["duplicate_quality"][0]\n        production = manifest["duplicate_diagnostics"]\n        require(quality["excess_copies"] == production["exact"]["excess"], "Duplicate diagnostics differ")\n        require(quality["feature_label_conflict_groups"] == production["cleaned_feature_conflicts"]["groups"], "Conflict diagnostics differ")\n        require(quality["rows_in_feature_label_conflicts"] == production["cleaned_feature_conflicts"]["participating_rows"], "Conflict rows differ")\n        require(quality["temporal_ineligible"] == validation["temporal_ineligible"], "Timestamp anomaly count differs")\n        verify_aggregates(data, validation)\n        validation["duplicate_aware_records"] = validation["records"] - quality["excess_copies"]\n        benchmark, plans, counts = compare_workload(view.select("source_day", "binary_label"), config["benchmark_repeats"])\n        require(sum(row["records"] for row in counts) == validation["records"], "Benchmark skipped records")\n        timings.extend(benchmark)\n        rdd = run_aggregation_demo(features.where("source_day = \'2018-03-01\'"))\n        atomic_json(output / "phase3_execution_plans.json", {"day_label_benchmark": plans, "bounded_rdd": rdd})\n        figures = draw(load_inputs(output), output / "figures")\n        validation.update(status="validated", elapsed_seconds=round(time.monotonic() - started, 3),\n                          figures=[item.name for item in figures], rdd_sample_rows=rdd["rows"],\n                          benchmark_repeats=config["benchmark_repeats"], quantile_accuracy=settings)\n        validation["resources"]["driver_peak_rss_bytes"] = driver_peak_bytes(spark)\n        write_checkpoint(output, validation, data, timings)\n        print(f"PHASE3 complete: {len(data)} queries, {len(figures)} charts; {validation[\'elapsed_seconds\']} s", flush=True)\n    finally:\n        if view is not None:\n            view.unpersist(blocking=True)\n        spark.stop()\n', 'src/visualization/eda_charts.py': '"""Render compact PNGs from Phase 3 aggregate CSVs."""\n\nimport argparse\nfrom collections import Counter\nimport csv\nimport json\nfrom pathlib import Path\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\n\nCOLORS = ("#007f83", "#b94c1d", "#5267a0", "#6b7e2e", "#955eb0", "#ba8275")\nFIGURES = (\n    "binary_distribution", "attack_family_frequencies", "daily_traffic_volume", "daily_attack_prevalence",\n    "family_by_capture_day", "flow_duration_comparison", "packet_byte_distributions", "protocol_distribution",\n    "destination_ports", "temporal_feature_changes",\n)\n\n\ndef read_table(root, name):\n    with (root / f"phase3_{name}.csv").open(encoding="utf-8", newline="") as stream:\n        return list(csv.DictReader(stream))\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef load_inputs(root):\n    tables = {name: read_table(root, name) for name in (\n        "traffic_totals", "attack_families", "daily_traffic", "family_by_day",\n        "feature_statistics", "protocols", "destination_ports", "time_windows",\n    )}\n    totals = {row["population"]: int(row["records"]) for row in tables["traffic_totals"]}\n    require(set(totals) == {"attack", "benign"}, "Expected both populations")\n    daily = tables["daily_traffic"]\n    require(sum(int(row["records"]) for row in daily) == sum(totals.values()), "Daily traffic total differs")\n    require(all(int(row["benign"]) + int(row["attacks"]) == int(row["records"]) for row in daily), "Daily label counts differ")\n    require(sum(int(row["records"]) for row in tables["attack_families"]) == totals["attack"], "Attack family total differs")\n    require(sum(int(row["records"]) for row in tables["family_by_day"]) == totals["attack"], "Family/day total differs")\n    require(sum(int(row["records"]) for row in tables["protocols"]) == sum(totals.values()), "Protocol total differs")\n    stats = tables["feature_statistics"]\n    required = {(name, population) for name in ("flow_duration", "total_packets", "total_bytes") for population in totals}\n    available = {(row["feature"], row["population"]) for row in stats if row["valid"] and row["quantiles"]}\n    require(required <= available, "Quantile chart input is incomplete")\n    return tables\n\n\ndef draw(tables, destination):\n    destination.mkdir(parents=True, exist_ok=True)\n    plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,\n                        "axes.grid": True, "grid.alpha": 0.20, "axes.axisbelow": True})\n    outputs = []\n\n    def save(fig, name):\n        path = destination / f"phase3_{name}.png"\n        fig.savefig(path, dpi=130, bbox_inches="tight", metadata={"Software": "MMDS Phase 3"})\n        plt.close(fig)\n        outputs.append(path)\n\n    def bar(name, values, ylabel, title, rotation=0, horizontal=False):\n        labels, numbers = zip(*values)\n        fig, ax = plt.subplots(figsize=(7.0, 3.4), layout="constrained")\n        if horizontal:\n            ax.barh(labels, numbers, color=COLORS[0])\n            ax.set_xlabel(ylabel)\n            ax.invert_yaxis()\n        else:\n            ax.bar(labels, numbers, color=[COLORS[i % len(COLORS)] for i in range(len(labels))])\n            ax.set_ylabel(ylabel)\n            ax.tick_params(axis="x", labelrotation=rotation)\n        ax.set_title(title, loc="left")\n        save(fig, name)\n\n    bar(FIGURES[0], [(row["population"].title(), int(row["records"])) for row in tables["traffic_totals"]],\n        "Records", "All captures · binary label distribution")\n    bar(FIGURES[1], [(row["attack_family"], int(row["records"])) for row in tables["attack_families"]],\n        "Attack records", "All captures · attack-family frequencies", horizontal=True)\n    daily = tables["daily_traffic"]\n    dates = [row["source_day"][5:] for row in daily]\n    bar(FIGURES[2], list(zip(dates, [int(row["records"]) for row in daily])), "Records", "Traffic volume by capture day", 30)\n    bar(FIGURES[3], list(zip(dates, [100 * float(row["attack_share"]) for row in daily])), "Attack share (%)",\n        "Attack prevalence by capture day", 30)\n\n    values = {(row["source_day"][5:], row["attack_family"]): int(row["records"]) for row in tables["family_by_day"]}\n    fig, ax = plt.subplots(figsize=(8.1, 4.3), layout="constrained")\n    bottom = [0] * len(dates)\n    for i, row in enumerate(tables["attack_families"]):\n        family = row["attack_family"]\n        counts = [values[(day, family)] for day in dates]\n        ax.bar(dates, counts, bottom=bottom, label=family, color=COLORS[i])\n        bottom = [a + b for a, b in zip(bottom, counts)]\n    ax.set_ylabel("Attack records")\n    ax.set_title("Attack families by capture day", loc="left")\n    ax.tick_params(axis="x", labelrotation=30)\n    ax.legend(ncols=2, fontsize=8)\n    save(fig, FIGURES[4])\n\n    stats = {(row["population"], row["feature"]): json.loads(row["quantiles"]) for row in tables["feature_statistics"]}\n\n    def quantile_axis(ax, feature, ylabel):\n        for i, population in enumerate(("benign", "attack")):\n            ax.plot(("P10", "P50", "P90", "P99"), stats[population, feature], marker="o",\n                    color=COLORS[i], label=population.title())\n        ax.set_yscale("symlog", linthresh=1)\n        ax.set_ylabel(ylabel)\n        ax.set_xlabel("Approximate quantile")\n        ax.legend()\n\n    fig, ax = plt.subplots(figsize=(6.2, 3.6), layout="constrained")\n    quantile_axis(ax, "flow_duration", "Duration (µs, symlog)")\n    ax.set_title("Flow duration by binary label", loc="left")\n    save(fig, FIGURES[5])\n\n    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.8), layout="constrained")\n    quantile_axis(axes[0], "total_packets", "Packets per flow (symlog)")\n    quantile_axis(axes[1], "total_bytes", "Bytes per flow (symlog)")\n    fig.suptitle("Packet and byte distributions by binary label")\n    save(fig, FIGURES[6])\n\n    protocols = Counter()\n    for row in tables["protocols"]:\n        protocols[row["protocol"]] += int(row["records"])\n    bar(FIGURES[7], sorted(protocols.items()), "Records", "All captures · IP protocol codes")\n\n    fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.4), layout="constrained")\n    for i, population in enumerate(("benign", "attack")):\n        counts = Counter()\n        for row in tables["destination_ports"]:\n            if row["population"] == population and row["scope"] == "population":\n                counts[row["dst_port"] or "Missing"] += int(row["records"])\n        top = counts.most_common(8)\n        axes[i].barh([port for port, _ in top], [count for _, count in top], color=COLORS[i])\n        axes[i].invert_yaxis()\n        axes[i].set_title(population.title())\n        axes[i].set_xlabel("Records")\n    fig.suptitle("Most frequent destination ports by label")\n    save(fig, FIGURES[8])\n\n    fig, ax = plt.subplots(figsize=(8.0, 4.0), layout="constrained")\n    times = sorted({(r["source_day"], int(r["window_start_hour"])) for r in tables["time_windows"]})\n    index = {key: i for i, key in enumerate(times)}\n    for i, population in enumerate(("benign", "attack")):\n        selected = [r for r in tables["time_windows"] if r["population"] == population and r["median_duration_us"]]\n        ax.scatter([index[r["source_day"], int(r["window_start_hour"])] for r in selected],\n                   [float(r["median_duration_us"]) for r in selected], s=13, color=COLORS[i], label=population.title())\n    ticks = [i for i, key in enumerate(times) if i == 0 or key[0] != times[i - 1][0]]\n    ax.set_xticks(ticks, [times[i][0][5:] for i in ticks], rotation=30)\n    ax.set_xlabel("Observed capture days · three-hour wall-clock windows")\n    ax.set_ylabel("Approximate median duration (µs, symlog)")\n    ax.set_yscale("symlog", linthresh=1)\n    ax.set_title("Eligible event-time flow duration summaries", loc="left")\n    ax.legend()\n    save(fig, FIGURES[9])\n    return outputs\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("tables", type=Path)\n    parser.add_argument("destination", type=Path)\n    args = parser.parse_args()\n    draw(load_inputs(args.tables), args.destination)\n', 'configs/analytics.yaml': '{\n  "version": "phase3-v1",\n  "quantile_accuracy": 4096,\n  "top_ports": 12,\n  "max_compact_rows": 6000,\n  "benchmark_repeats": 3,\n  "spark": {"master": "local[4]", "driver_memory": "8g", "shuffle_partitions": 48, "input_partition_bytes": 67108864},\n  "flow_statistics": [\n    "flow_duration", "total_packets", "total_bytes", "tot_fwd_pkts", "tot_bwd_pkts",\n    "totlen_fwd_pkts", "totlen_bwd_pkts", "pkt_len_mean", "flow_iat_mean",\n    "flow_pkts_s", "flow_byts_s", "syn_flag_cnt", "ack_flag_cnt", "rst_flag_cnt"\n  ],\n  "correlations": [\n    ["flow_duration", "total_packets"], ["flow_duration", "total_bytes"],\n    ["tot_fwd_pkts", "tot_bwd_pkts"], ["totlen_fwd_pkts", "totlen_bwd_pkts"],\n    ["total_packets", "total_bytes"], ["pkt_len_mean", "total_bytes"],\n    ["flow_iat_mean", "flow_duration"], ["syn_flag_cnt", "rst_flag_cnt"]\n  ]\n}\n', 'configs/cleaning.yaml': '{\n  "version": "phase2-v1",\n  "dataset_version": 1,\n  "headers": {\n    "common_80": [\n      "Dst Port",\n      "Protocol",\n      "Timestamp",\n      "Flow Duration",\n      "Tot Fwd Pkts",\n      "Tot Bwd Pkts",\n      "TotLen Fwd Pkts",\n      "TotLen Bwd Pkts",\n      "Fwd Pkt Len Max",\n      "Fwd Pkt Len Min",\n      "Fwd Pkt Len Mean",\n      "Fwd Pkt Len Std",\n      "Bwd Pkt Len Max",\n      "Bwd Pkt Len Min",\n      "Bwd Pkt Len Mean",\n      "Bwd Pkt Len Std",\n      "Flow Byts/s",\n      "Flow Pkts/s",\n      "Flow IAT Mean",\n      "Flow IAT Std",\n      "Flow IAT Max",\n      "Flow IAT Min",\n      "Fwd IAT Tot",\n      "Fwd IAT Mean",\n      "Fwd IAT Std",\n      "Fwd IAT Max",\n      "Fwd IAT Min",\n      "Bwd IAT Tot",\n      "Bwd IAT Mean",\n      "Bwd IAT Std",\n      "Bwd IAT Max",\n      "Bwd IAT Min",\n      "Fwd PSH Flags",\n      "Bwd PSH Flags",\n      "Fwd URG Flags",\n      "Bwd URG Flags",\n      "Fwd Header Len",\n      "Bwd Header Len",\n      "Fwd Pkts/s",\n      "Bwd Pkts/s",\n      "Pkt Len Min",\n      "Pkt Len Max",\n      "Pkt Len Mean",\n      "Pkt Len Std",\n      "Pkt Len Var",\n      "FIN Flag Cnt",\n      "SYN Flag Cnt",\n      "RST Flag Cnt",\n      "PSH Flag Cnt",\n      "ACK Flag Cnt",\n      "URG Flag Cnt",\n      "CWE Flag Count",\n      "ECE Flag Cnt",\n      "Down/Up Ratio",\n      "Pkt Size Avg",\n      "Fwd Seg Size Avg",\n      "Bwd Seg Size Avg",\n      "Fwd Byts/b Avg",\n      "Fwd Pkts/b Avg",\n      "Fwd Blk Rate Avg",\n      "Bwd Byts/b Avg",\n      "Bwd Pkts/b Avg",\n      "Bwd Blk Rate Avg",\n      "Subflow Fwd Pkts",\n      "Subflow Fwd Byts",\n      "Subflow Bwd Pkts",\n      "Subflow Bwd Byts",\n      "Init Fwd Win Byts",\n      "Init Bwd Win Byts",\n      "Fwd Act Data Pkts",\n      "Fwd Seg Size Min",\n      "Active Mean",\n      "Active Std",\n      "Active Max",\n      "Active Min",\n      "Idle Mean",\n      "Idle Std",\n      "Idle Max",\n      "Idle Min",\n      "Label"\n    ],\n    "extended_84": [\n      "Flow ID",\n      "Src IP",\n      "Src Port",\n      "Dst IP",\n      "Dst Port",\n      "Protocol",\n      "Timestamp",\n      "Flow Duration",\n      "Tot Fwd Pkts",\n      "Tot Bwd Pkts",\n      "TotLen Fwd Pkts",\n      "TotLen Bwd Pkts",\n      "Fwd Pkt Len Max",\n      "Fwd Pkt Len Min",\n      "Fwd Pkt Len Mean",\n      "Fwd Pkt Len Std",\n      "Bwd Pkt Len Max",\n      "Bwd Pkt Len Min",\n      "Bwd Pkt Len Mean",\n      "Bwd Pkt Len Std",\n      "Flow Byts/s",\n      "Flow Pkts/s",\n      "Flow IAT Mean",\n      "Flow IAT Std",\n      "Flow IAT Max",\n      "Flow IAT Min",\n      "Fwd IAT Tot",\n      "Fwd IAT Mean",\n      "Fwd IAT Std",\n      "Fwd IAT Max",\n      "Fwd IAT Min",\n      "Bwd IAT Tot",\n      "Bwd IAT Mean",\n      "Bwd IAT Std",\n      "Bwd IAT Max",\n      "Bwd IAT Min",\n      "Fwd PSH Flags",\n      "Bwd PSH Flags",\n      "Fwd URG Flags",\n      "Bwd URG Flags",\n      "Fwd Header Len",\n      "Bwd Header Len",\n      "Fwd Pkts/s",\n      "Bwd Pkts/s",\n      "Pkt Len Min",\n      "Pkt Len Max",\n      "Pkt Len Mean",\n      "Pkt Len Std",\n      "Pkt Len Var",\n      "FIN Flag Cnt",\n      "SYN Flag Cnt",\n      "RST Flag Cnt",\n      "PSH Flag Cnt",\n      "ACK Flag Cnt",\n      "URG Flag Cnt",\n      "CWE Flag Count",\n      "ECE Flag Cnt",\n      "Down/Up Ratio",\n      "Pkt Size Avg",\n      "Fwd Seg Size Avg",\n      "Bwd Seg Size Avg",\n      "Fwd Byts/b Avg",\n      "Fwd Pkts/b Avg",\n      "Fwd Blk Rate Avg",\n      "Bwd Byts/b Avg",\n      "Bwd Pkts/b Avg",\n      "Bwd Blk Rate Avg",\n      "Subflow Fwd Pkts",\n      "Subflow Fwd Byts",\n      "Subflow Bwd Pkts",\n      "Subflow Bwd Byts",\n      "Init Fwd Win Byts",\n      "Init Bwd Win Byts",\n      "Fwd Act Data Pkts",\n      "Fwd Seg Size Min",\n      "Active Mean",\n      "Active Std",\n      "Active Max",\n      "Active Min",\n      "Idle Mean",\n      "Idle Std",\n      "Idle Max",\n      "Idle Min",\n      "Label"\n    ]\n  },\n  "extended_day": "2018-02-20",\n  "label_mapping": {\n    "Benign": {\n      "normalized": "benign",\n      "binary": 0,\n      "family": "Benign"\n    },\n    "Bot": {\n      "normalized": "bot",\n      "binary": 1,\n      "family": "Bot"\n    },\n    "Brute Force -Web": {\n      "normalized": "brute force -web",\n      "binary": 1,\n      "family": "Web attack"\n    },\n    "Brute Force -XSS": {\n      "normalized": "brute force -xss",\n      "binary": 1,\n      "family": "Web attack"\n    },\n    "DDOS attack-HOIC": {\n      "normalized": "ddos attack-hoic",\n      "binary": 1,\n      "family": "DDoS"\n    },\n    "DDOS attack-LOIC-UDP": {\n      "normalized": "ddos attack-loic-udp",\n      "binary": 1,\n      "family": "DDoS"\n    },\n    "DDoS attacks-LOIC-HTTP": {\n      "normalized": "ddos attacks-loic-http",\n      "binary": 1,\n      "family": "DDoS"\n    },\n    "DoS attacks-GoldenEye": {\n      "normalized": "dos attacks-goldeneye",\n      "binary": 1,\n      "family": "DoS"\n    },\n    "DoS attacks-Hulk": {\n      "normalized": "dos attacks-hulk",\n      "binary": 1,\n      "family": "DoS"\n    },\n    "DoS attacks-SlowHTTPTest": {\n      "normalized": "dos attacks-slowhttptest",\n      "binary": 1,\n      "family": "DoS"\n    },\n    "DoS attacks-Slowloris": {\n      "normalized": "dos attacks-slowloris",\n      "binary": 1,\n      "family": "DoS"\n    },\n    "FTP-BruteForce": {\n      "normalized": "ftp-bruteforce",\n      "binary": 1,\n      "family": "Credential brute force"\n    },\n    "Infilteration": {\n      "normalized": "infilteration",\n      "binary": 1,\n      "family": "Infiltration"\n    },\n    "SQL Injection": {\n      "normalized": "sql injection",\n      "binary": 1,\n      "family": "Web attack"\n    },\n    "SSH-Bruteforce": {\n      "normalized": "ssh-bruteforce",\n      "binary": 1,\n      "family": "Credential brute force"\n    }\n  },\n  "sentinel_values": {\n    "init_fwd_win_byts": [\n      -1\n    ],\n    "init_bwd_win_byts": [\n      -1\n    ]\n  },\n  "numeric_actions": {\n    "nan": "nullify",\n    "positive_infinity": "nullify",\n    "negative_infinity": "nullify",\n    "invalid_negative": "nullify",\n    "sentinel": "nullify",\n    "cast_failure": "typed_null",\n    "source_null": "retain_null"\n  },\n  "excluded_shared_features": [\n    "dst_port"\n  ],\n  "duplicate_policy": "diagnostic",\n  "expected_full": {\n    "benign": 13484708,\n    "attack": 2748235,\n    "nan": 59721,\n    "infinity": 131799,\n    "invalid_negative": 106,\n    "timestamp_outlier": 14,\n    "duplicate_excess": 410707\n  },\n  "spark": {\n    "master": "local[4]",\n    "driver_memory": "8g",\n    "input_partition_bytes": 67108864,\n    "shuffle_partitions": 48\n  },\n  "target_source_bytes_per_part": 134217728\n}\n', 'manifests/parquet_layers.json': '{\n  "status": "validated",\n  "dataset": {\n    "owner": "saisathwikregula",\n    "slug": "mmds-dataset-23013",\n    "version": 1\n  },\n  "policy_version": "phase2-v1",\n  "layers": {\n    "raw_aligned": {\n      "path": "/kaggle/working/mmds_phase2/phase2-v1/raw_aligned",\n      "partition_column": "source_day",\n      "partitions": 10,\n      "format": "parquet",\n      "compression": "SNAPPY",\n      "records": 16232943,\n      "bytes": 2712371594,\n      "files": 55,\n      "write_seconds": 240.044\n    },\n    "clean": {\n      "path": "/kaggle/working/mmds_phase2/phase2-v1/clean",\n      "partition_column": "source_day",\n      "partitions": 10,\n      "format": "parquet",\n      "compression": "SNAPPY",\n      "records": 16232943,\n      "bytes": 2708993526,\n      "files": 55,\n      "write_seconds": 297.806\n    },\n    "feature_ready": {\n      "path": "/kaggle/working/mmds_phase2/phase2-v1/feature_ready",\n      "partition_column": "source_day",\n      "partitions": 10,\n      "format": "parquet",\n      "compression": "SNAPPY",\n      "records": 16232943,\n      "bytes": 3003003600,\n      "files": 53,\n      "write_seconds": 314.859\n    }\n  },\n  "duplicate_policy": "diagnostic",\n  "duplicate_diagnostics": {\n    "records_removed_primary": 0,\n    "exact": {\n      "groups": 107670,\n      "excess": 410707\n    },\n    "cleaned_feature_conflicts": {\n      "groups": 21763,\n      "participating_rows": 710014\n    },\n    "diagnostics_path": "/kaggle/working/mmds_phase2/phase2-v1/diagnostics",\n    "seconds": 72.76\n  },\n  "producer": {\n    "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n    "kernel_version": 1,\n    "status": "COMPLETE"\n  },\n  "resources": {\n    "spark_version": "4.3.0",\n    "master": "local[4]",\n    "driver_memory": "8g",\n    "input_partition_bytes": 67108864,\n    "shuffle_partitions": 48,\n    "default_parallelism": 4,\n    "cpu_affinity": 4,\n    "scratch_output_free_bytes_start": 20940369920,\n    "memory.max": "32212254720",\n    "cpu.max": "400000 100000",\n    "driver_peak_rss_bytes": 5784305664\n  },\n  "totals": {\n    "physical_lines": 16233012,\n    "parsed": 16232943,\n    "headers": 69,\n    "blank": 0,\n    "bad_width_or_quotes": 0,\n    "spark_corrupt": 0,\n    "repeated_headers": 59,\n    "canonical_rows": 16232943,\n    "benign": 13484708,\n    "attack": 2748235,\n    "unknown_labels": 0,\n    "timestamp_outlier": 14,\n    "timestamp_unparseable": 0,\n    "timestamp_missing": 0,\n    "temporal_eligible": 16232929,\n    "quality_affected_rows": 8257114,\n    "quality_affected_cells": 12879783,\n    "structural_missing_rows": 8284195,\n    "structural_missing_cells": 33136780,\n    "nan_rows": 59721,\n    "nan_cells": 59721,\n    "positive_infinity_rows": 95760,\n    "positive_infinity_cells": 131799,\n    "negative_infinity_rows": 0,\n    "negative_infinity_cells": 0,\n    "invalid_negative_rows": 15,\n    "invalid_negative_cells": 106,\n    "sentinel_rows": 8255549,\n    "sentinel_cells": 12688157,\n    "cast_failure_rows": 0,\n    "cast_failure_cells": 0,\n    "source_null_rows": 0,\n    "source_null_cells": 0\n  },\n  "independent_mount": {\n    "layers": {\n      "raw_aligned": {\n        "records": 16232943,\n        "bytes": 2712371594,\n        "benign": 13484708,\n        "attack": 2748235,\n        "files": 55,\n        "partitions": 10\n      },\n      "clean": {\n        "records": 16232943,\n        "bytes": 2708993526,\n        "benign": 13484708,\n        "attack": 2748235,\n        "files": 55,\n        "partitions": 10\n      },\n      "feature_ready": {\n        "records": 16232943,\n        "bytes": 3003003600,\n        "benign": 13484708,\n        "attack": 2748235,\n        "files": 53,\n        "partitions": 10\n      }\n    },\n    "clean_cells_verified_null": 12879783,\n    "clean_quality_affected_rows": 8257114,\n    "clean_quality_reasons": {\n      "sentinel": 12688157,\n      "nan": 59721,\n      "positive_infinity": 131799,\n      "invalid_negative": 106\n    },\n    "clean_timestamp_outliers": 14,\n    "summary_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n    "producer_mount": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2",\n    "elapsed_seconds": 101.019,\n    "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-parquet-reuse-check",\n    "kernel_version": 1,\n    "status": "COMPLETE"\n  },\n  "feature_columns": [\n    "protocol",\n    "flow_duration",\n    "tot_fwd_pkts",\n    "tot_bwd_pkts",\n    "totlen_fwd_pkts",\n    "totlen_bwd_pkts",\n    "fwd_pkt_len_max",\n    "fwd_pkt_len_min",\n    "fwd_pkt_len_mean",\n    "fwd_pkt_len_std",\n    "bwd_pkt_len_max",\n    "bwd_pkt_len_min",\n    "bwd_pkt_len_mean",\n    "bwd_pkt_len_std",\n    "flow_byts_s",\n    "flow_pkts_s",\n    "flow_iat_mean",\n    "flow_iat_std",\n    "flow_iat_max",\n    "flow_iat_min",\n    "fwd_iat_tot",\n    "fwd_iat_mean",\n    "fwd_iat_std",\n    "fwd_iat_max",\n    "fwd_iat_min",\n    "bwd_iat_tot",\n    "bwd_iat_mean",\n    "bwd_iat_std",\n    "bwd_iat_max",\n    "bwd_iat_min",\n    "fwd_psh_flags",\n    "bwd_psh_flags",\n    "fwd_urg_flags",\n    "bwd_urg_flags",\n    "fwd_header_len",\n    "bwd_header_len",\n    "fwd_pkts_s",\n    "bwd_pkts_s",\n    "pkt_len_min",\n    "pkt_len_max",\n    "pkt_len_mean",\n    "pkt_len_std",\n    "pkt_len_var",\n    "fin_flag_cnt",\n    "syn_flag_cnt",\n    "rst_flag_cnt",\n    "psh_flag_cnt",\n    "ack_flag_cnt",\n    "urg_flag_cnt",\n    "cwe_flag_count",\n    "ece_flag_cnt",\n    "down_up_ratio",\n    "pkt_size_avg",\n    "fwd_seg_size_avg",\n    "bwd_seg_size_avg",\n    "fwd_byts_b_avg",\n    "fwd_pkts_b_avg",\n    "fwd_blk_rate_avg",\n    "bwd_byts_b_avg",\n    "bwd_pkts_b_avg",\n    "bwd_blk_rate_avg",\n    "subflow_fwd_pkts",\n    "subflow_fwd_byts",\n    "subflow_bwd_pkts",\n    "subflow_bwd_byts",\n    "init_fwd_win_byts",\n    "init_bwd_win_byts",\n    "fwd_act_data_pkts",\n    "fwd_seg_size_min",\n    "active_mean",\n    "active_std",\n    "active_max",\n    "active_min",\n    "idle_mean",\n    "idle_std",\n    "idle_max",\n    "idle_min"\n  ]\n}\n'}
for relative, source in sources.items():
    target = project / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
sys.path.insert(0, str(project))


In [ ]:
from src.sql.phase3_pipeline import run
run(project, Path("/kaggle/input"), Path("/kaggle/working/mmds_phase3"),
    Path("/kaggle/temp/mmds_phase3/spark"))
